# Unet model

In [ ]:
!pip install -q --no-deps segmentation_models_pytorch ttach
!pip install -q albumentations

In [ ]:
import os
import random
import glob
import time
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjust to your Kaggle dataset name
DATA_ROOT   = Path('/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release' )  
IMAGE_DIR   = Path('/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_images')
MASK_DIR    = Path('/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_masks')
OUTPUT_DIR  = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Image ─────────────────────────────────────────────────────────────────────
IMG_SIZE    = 512          # both height and width
IMG_H, IMG_W = 480, 640  
IMG_EXT     = '.jpg'       # extension of raw images
MASK_EXT    = '.png'       # extension of mask files

# ── Split ─────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.80         # 80 % train / 10 % val / 10 % test
VAL_RATIO   = 0.10

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE      = 8
NUM_WORKERS     = 0
LR              = 1e-4
WEIGHT_DECAY    = 1e-5
MAX_EPOCHS      = 30
PATIENCE        = 10       # early-stopping patience (val epochs without improvement)
DICE_WEIGHT     = 0.5
BCE_WEIGHT      = 0.5

# ── Model ─────────────────────────────────────────────────────────────────────
ENCODER         = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
CLASSES         = 1        # binary segmentation

# ── Post-processing ───────────────────────────────────────────────────────────
THRESHOLD       = 0.5
MIN_AREA        = 500      # px² — small connected components below this are removed

print('Config OK')

In [ ]:
# ── Collect matched pairs ──────────────────────────────────────────────────────
import glob as glob_module
image_paths  = sorted(glob_module.glob(os.path.join(IMAGE_DIR,  '*.jpg')))
mask_paths = sorted(glob_module.glob(os.path.join(MASK_DIR, '*.png')))
print(f"Images trouvées : {len(image_paths)}")
print(f"Masques trouvés : {len(mask_paths)}")
N=len(mask_paths)
# ── Reproducible shuffle & split ──────────────────────────────────────────────
rng     = np.random.default_rng(SEED)
indices = rng.permutation(N)

n_train = int(N * TRAIN_RATIO)
n_val   = int(N * VAL_RATIO)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_images = [image_paths[i] for i in train_idx]
train_masks  = [mask_paths[i]  for i in train_idx]
val_images   = [image_paths[i] for i in val_idx]
val_masks    = [mask_paths[i]  for i in val_idx]
test_images  = [image_paths[i] for i in test_idx]
test_masks   = [mask_paths[i]  for i in test_idx]

print(f'Train : {len(train_images):>4}  ({len(train_images)/N*100:.0f}%)')
print(f'Val   : {len(val_images):>4}  ({len(val_images)/N*100:.0f}%)')
print(f'Test  : {len(test_images):>4}  ({len(test_images)/N*100:.0f}%)')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
# Training augmentations — mirrors the FHDO team's Albumentations pipeline (Section 4.5.1)
train_transform = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.HorizontalFlip(p=0.5),           
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(p=0.5),
    A.OneOf([A.GridDistortion(), A.ElasticTransform()], p=0.5),
    A.OneOf([A.CLAHE(), A.RandomGamma(), A.RandomBrightnessContrast()], p=0.5),
    A.GaussNoise(p=0.3),
    A.Normalize(),                     # ImageNet mean/std (needed for pretrained backbone)
    ToTensorV2(),
])

# Validation / test — no geometric changes, just resize + normalise
val_tfm = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('Augmentation pipelines defined.')

test this cell for agressive augmentation

In [ ]:
# ── CELL A1 : Aggressive augmentation pipeline ────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# This is the NEW pipeline to compare against your current baseline.
# Each group is additive on top of the minimal pipeline.
# Supervisor's note: DFU photos suffer from lighting, motion blur, angle variance
# → we simulate all of these explicitly.
# Mean/std from ImageNet (used to normalise before feeding pretrained encoder)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
train_tfm_aggressive = A.Compose([

    # ── Group 1 : Geometry ────────────────────────────────────────────────────
    # Same as baseline but with wider rotate_limit and stronger elastic deform.
    # Rationale: wound shape is rotation/scale invariant; the model should not
    # overfit to the specific camera angle of the dataset.
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.10,       # wider shift than baseline (0.05 → 0.10)
        scale_limit=0.20,       # wider scale  than baseline (0.10 → 0.20)
        rotate_limit=30,        # wider angle  than baseline (15   → 30)
        border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6
    ),
    A.ElasticTransform(alpha=120, sigma=12, p=0.35),  # stronger deformation
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),

    # ── Group 2 : Color / exposure ────────────────────────────────────────────
    # DFU images are taken in clinics under variable lighting.
    # We simulate: overexposure, underexposure, wrong white balance,
    # and contrast stretching (CLAHE is common in medical imaging).
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=1.0),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=40, val_shift_limit=30, p=1.0),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomGamma(gamma_limit=(70, 140), p=1.0),  # NEW: gamma exposure shift
    ], p=0.75),                  # higher probability than baseline (0.6 → 0.75)

    # ── Group 3 : Blur / motion / noise ───────────────────────────────────────
    # Simulates: camera shake (MotionBlur), out-of-focus macro (GaussianBlur),
    # sensor noise (GaussNoise), JPEG compression artifacts (ImageCompression).
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=(3, 9), p=1.0),       # NEW: motion blur
        A.MedianBlur(blur_limit=5, p=1.0),             # NEW: smoothing artifact
    ], p=0.40),
    A.GaussNoise(var_limit=(15, 65), p=0.35),          # stronger than baseline
    A.ImageCompression(quality_lower=60, quality_upper=95, p=0.25),  # NEW: JPEG

    # ── Group 4 : Occlusion / dropout ────────────────────────────────────────
    # CoarseDropout simulates partial occlusion (bandage edges, tape, gloves).
    # More holes and larger size than baseline to be more aggressive.
    A.CoarseDropout(
        max_holes=8,            # baseline: 4
        max_height=48,          # baseline: 32
        max_width=48,           # baseline: 32
        min_holes=1,
        fill_value=0,
        mask_fill_value=0,      # mask pixels under dropout stay 0 (not wound)
        p=0.30
    ),

    # ── Normalise + tensor ────────────────────────────────────────────────────
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


# ── Validation / test augmentation (resize + normalise only) ──────────────────
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
print('Aggressive pipeline defined — ready for ablation.')

test this cell for no augmentation

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
train_transform = A.Compose([
    A.Resize(512,512),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
print('Aggressive pipeline defined — ready for ablation.')

In [ ]:
class WoundDataset(Dataset):
    """
    Loads (image, mask) pairs.
    Mask pixels >= 128 are treated as wound (binary 1).
    """
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image (BGR → RGB)
        image = cv2.imread(str(self.image_paths[idx]))
        if image is None:
            raise FileNotFoundError(f'Cannot load image: {self.image_paths[idx]}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load mask (grayscale)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f'Cannot load mask: {self.mask_paths[idx]}')

        # Binarise: wound = 1
        mask = (mask >= 128).astype(np.float32)

        if self.transform:
            aug  = self.transform(image=image, mask=mask)
            image = aug['image']          # (C, H, W) float32 tensor
            mask  = aug['mask']           # (H, W)    float32 tensor

        # Add channel dim to mask → (1, H, W)
        mask = mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.tensor(mask).unsqueeze(0)

        return image, mask


# ── DataLoaders ───────────────────────────────────────────────────────────────
train_dataset = WoundDataset(train_images, train_masks, transform=train_transform)
val_dataset   = WoundDataset(val_images,   val_masks,   transform=val_transform)
test_dataset  = WoundDataset(test_images,  test_masks,  transform=val_transform)

train_loader  = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader    = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader   = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

# Quick shape sanity-check
imgs, masks = next(iter(train_loader))
print(f'Image batch shape : {imgs.shape}   dtype: {imgs.dtype}')
print(f'Mask  batch shape : {masks.shape}  dtype: {masks.dtype}')
print(f'Mask  unique vals : {masks.unique().tolist()}')

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalisation for a (C,H,W) tensor → (H,W,C) uint8 array."""
    t   = tensor.clone().cpu().float()
    m   = torch.tensor(mean).view(3, 1, 1)
    s   = torch.tensor(std).view(3, 1, 1)
    t   = t * s + m
    t   = torch.clamp(t, 0, 1)
    return (t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def visualize_batch_with_overlay(
    images: torch.Tensor,
    masks:  torch.Tensor,
    n_show: int  = 8,
    alpha:  float = 0.45,
    title:  str   = 'Augmented batch — mask overlay (red = wound)'
):
    """
    Shows `n_show` samples from the batch.
    Each column: original image | image with red wound overlay.
    """
    n_show  = min(n_show, images.shape[0])
    fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3, 6))

    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

    for i in range(n_show):
        img_np   = denormalize(images[i])                          # (H,W,3) uint8
        mask_np  = masks[i, 0].cpu().numpy()                       # (H,W)  float32 {0,1}

        # ── Top row: raw image ────────────────────────────────────────────────
        axes[0, i].imshow(img_np)
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Image', fontsize=9, color='#444', pad=4)

        # ── Bottom row: image + red mask overlay ──────────────────────────────
        overlay = img_np.copy()
        wound_pixels = mask_np > 0.5
        # Paint wound pixels red with transparency
        overlay[wound_pixels] = (
            alpha * np.array([220, 30, 30]) +
            (1 - alpha) * img_np[wound_pixels]
        ).astype(np.uint8)

        axes[1, i].imshow(overlay)
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('+ GT mask (red)', fontsize=9, color='#c00', pad=4)

        # Wound coverage %
        pct = wound_pixels.mean() * 100
        axes[1, i].set_xlabel(f'{pct:.1f}% wound', fontsize=8, color='#555')

    # Legend
    red_patch = mpatches.Patch(color=(220/255, 30/255, 30/255), label='Wound mask')
    fig.legend(handles=[red_patch], loc='lower center', ncol=1,
               fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'augmented_batch_overlay.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUT_DIR}/augmented_batch_overlay.png')


# ── Pull a fresh batch and visualise ─────────────────────────────────────────
imgs, msks = next(iter(train_loader))
visualize_batch_with_overlay(imgs, msks, n_show=8)

In [ ]:
# ── Combined loss: 0.5 * DiceLoss + 0.5 * BCEWithLogitsLoss ─────────────────
dice_loss_fn = DiceLoss(mode='binary', from_logits=True)
bce_loss_fn  = nn.BCEWithLogitsLoss()

def combined_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return DICE_WEIGHT * dice_loss_fn(logits, targets) + \
           BCE_WEIGHT  * bce_loss_fn(logits, targets)


# ── Dice metric (for monitoring, not differentiating) ─────────────────────────
def dice_score(logits: torch.Tensor, targets: torch.Tensor,
               threshold: float = 0.5, eps: float = 1e-6) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    inter = (preds * targets).sum()
    union = preds.sum() + targets.sum()
    return ((2.0 * inter + eps) / (union + eps)).item()


# ── Optimiser & scheduler ─────────────────────────────────────────────────────
optimizer = optim.AdamW(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode    = 'min',
    factor  = 0.5,
    patience = 5,
    min_lr  = 1e-7,
    
)

print('Loss, optimiser and scheduler ready.')

In [ ]:
model = smp.Unet(
    encoder_name    = ENCODER,
    encoder_weights = ENCODER_WEIGHTS,
    in_channels     = 3,
    classes         = CLASSES,
    activation      = None,     # raw logits — we apply sigmoid in loss / post-proc
)
model = model.to(DEVICE)

# Parameter count
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')

test this for focal loss

In [ ]:
# ── CELL 6 : Hybrid loss — Dice + Focal ──────────────────────────────────────
# Why Focal? It down-weights easy negatives (healthy skin pixels, which dominate)
# and forces the model to focus on hard boundary pixels — exactly where DFU
# segmentation fails most.
#
# Formula:
#   FL(p_t) = -α(1 - p_t)^γ * log(p_t)
#   γ=2   → quadratic penalty for confident correct predictions
#   α=0.25 → balances foreground (wound) vs background
#
# Combined: 0.5 * DiceLoss + 0.5 * FocalLoss

from segmentation_models_pytorch.losses import DiceLoss, FocalLoss

dice_loss_fn  = DiceLoss(mode='binary', from_logits=True)
focal_loss_fn = FocalLoss(
    mode       = 'binary',
    alpha      = 0.25,   # weight for the positive (wound) class
    gamma      = 2.0,    # focusing parameter — higher = more focus on hard pixels
    normalized = False,
)

DICE_WEIGHT  = 0.5
FOCAL_WEIGHT = 0.5

def dice_score(logits: torch.Tensor, targets: torch.Tensor,
               threshold: float = 0.5, eps: float = 1e-6) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    inter = (preds * targets).sum()
    union = preds.sum() + targets.sum()
    return ((2.0 * inter + eps) / (union + eps)).item() 
    
def combined_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Dice + Focal hybrid. Both losses work on raw logits."""
    return (DICE_WEIGHT  * dice_loss_fn(logits, targets) +
            FOCAL_WEIGHT * focal_loss_fn(logits, targets))

# Quick sanity check
_dummy_logits = torch.randn(2, 1, 512, 512)
_dummy_masks  = (torch.rand(2, 1, 512, 512) > 0.8).float()
_loss_val     = combined_loss(_dummy_logits, _dummy_masks)
print(f'Hybrid Dice+Focal loss (random input): {_loss_val.item():.4f}')
print('Loss function updated — re-run training cells to use it.')


optimizer = optim.AdamW(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode    = 'min',
    factor  = 0.5,
    patience = 5,
    min_lr  = 1e-7,
    
)

In [ ]:
from tqdm.auto import tqdm
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_dice = 0.0, 0.0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, masks in pbar:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss   = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        batch_dice = dice_score(logits.detach(), masks)

        total_loss += loss.item()
        total_dice += batch_dice

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "dice": f"{batch_dice:.4f}"
        })

    n = len(loader)
    return total_loss / n, total_dice / n
@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss, total_dice = 0.0, 0.0

    pbar = tqdm(loader, desc="Validation", leave=False)

    for images, masks in pbar:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        logits = model(images)
        loss   = combined_loss(logits, masks)

        batch_dice = dice_score(logits, masks)

        total_loss += loss.item()
        total_dice += batch_dice

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "dice": f"{batch_dice:.4f}"
        })

    n = len(loader)
    return total_loss / n, total_dice / n

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': []
}

best_val_loss     = float('inf')
best_val_dice     = 0.0
patience_counter  = 0
BEST_CKPT         = OUTPUT_DIR / 'best_model.pth'

print(f'{'Epoch':>6}  {'Train Loss':>10}  {'Train Dice':>10}  '
      f'{'Val Loss':>8}  {'Val Dice':>8}  {'LR':>8}  {'Time':>6}')
print('-' * 72)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_dice = train_one_epoch(model, train_loader, optimizer, DEVICE)
    vl_loss, vl_dice = validate(model, val_loader, DEVICE)

    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_dice'].append(tr_dice)
    history['val_dice'].append(vl_dice)

    elapsed = time.time() - t0
    cur_lr  = optimizer.param_groups[0]['lr']

    improved = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_val_dice = vl_dice
        patience_counter = 0
        torch.save(model.state_dict(), BEST_CKPT)
        improved = ' ✓ saved'
    else:
        patience_counter += 1

    print(f'{epoch:>6}  {tr_loss:>10.4f}  {tr_dice:>10.4f}  '
          f'{vl_loss:>8.4f}  {vl_dice:>8.4f}  {cur_lr:>8.2e}  '
          f'{elapsed:>5.1f}s{improved}')

    if patience_counter >= PATIENCE:
        print(f'\n⏹  Early stopping triggered after {epoch} epochs '
              f'(no improvement for {PATIENCE} consecutive epochs).')
        break
 
print(f'\nBest val loss : {best_val_loss:.4f}  |  Best val Dice : {best_val_dice:.4f}')
print(f'Checkpoint    : {BEST_CKPT}') 

In [ ]:
# ── CELL 1 : Training curves ──────────────────────────────────────────────────
import matplotlib.pyplot as plt    
import matplotlib.patches as mpatches           
                 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))     
fig.suptitle('Training history', fontsize=14, fontweight='bold')

epochs = range(1, len(history['train_loss']) + 1)
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1

for ax, metric, title in zip(
    axes,
    [('train_loss', 'val_loss'), ('train_dice', 'val_dice')],
    ['Loss (train vs val)', 'Dice score (train vs val)']
):
    ax.plot(epochs, history[metric[0]], label='Train', color='steelblue', linewidth=1.8)
    ax.plot(epochs, history[metric[1]], label='Val',   color='coral',     linewidth=1.8)
    ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1,
               label=f'Best (epoch {best_epoch})') 
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_DIR}/training_curves.png')

In [ ]:
# ── CELL 2 : Visualisation — image / predicted mask / ground truth ────────────
import torch, matplotlib.pyplot as plt
import numpy as np

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def denorm(tensor):
    """Undo ImageNet normalisation for display."""
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

model.eval()
# Load best checkpoint before visualising
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))

N_SHOW = 4   # number of samples to display
fig, axes = plt.subplots(N_SHOW, 3, figsize=(12, 4 * N_SHOW))
fig.suptitle('Image  |  Ground truth  |  Prediction', fontsize=13, fontweight='bold')
col_titles = ['Image', 'Ground truth', 'Predicted mask']

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

batch_imgs, batch_masks = next(iter(val_loader))

with torch.no_grad():
    logits = model(batch_imgs.to(DEVICE))
    probs  = torch.sigmoid(logits).cpu()
    preds  = (probs > THRESHOLD).float()

for i in range(N_SHOW):
    for ax, col in zip(axes[i], col_titles):
        ax.axis('off')
        if i == 0:
            ax.set_title(col, fontsize=11)
    axes[i, 0].imshow(denorm(batch_imgs[i]))
    axes[i, 1].imshow(batch_masks[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 2].imshow(preds[i].squeeze(),       cmap='gray', vmin=0, vmax=1)
    # per-sample dice
    d = dice_score(logits[i:i+1].cpu(), batch_masks[i:i+1])
    axes[i, 2].set_title(f'Dice = {d:.3f}', fontsize=9, color='green' if d > 0.6 else 'red')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'predictions_vs_gt.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 3 : Step-by-step decoding of one prediction ─────────────────────────
model.eval()
sample_img, sample_mask = val_loader.dataset[0]
sample_img = sample_img.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    raw_logit  = model(sample_img)               # raw logit map
    prob_map   = torch.sigmoid(raw_logit)        # sigmoid → [0,1]
    binary_map = (prob_map > THRESHOLD).float()  # threshold → binary

steps = [
    (denorm(sample_img.squeeze(0).cpu()),  'Step 0 — Input image',  None),
    (raw_logit.squeeze().cpu().numpy(),    'Step 1 — Raw logits',   'RdBu_r'),
    (prob_map.squeeze().cpu().numpy(),     'Step 2 — Sigmoid probs','viridis'),
    (binary_map.squeeze().cpu().numpy(),   'Step 3 — Binary mask',  'gray'),
    (sample_mask.squeeze().numpy(),        'Ground truth',           'gray'),
]

fig, axes = plt.subplots(1, len(steps), figsize=(4 * len(steps), 4))
fig.suptitle('Decoding pipeline: logit → sigmoid → binary', fontsize=12, fontweight='bold')

for ax, (img, title, cmap) in zip(axes, steps):
    if cmap is None:
        ax.imshow(img)
    else:
        im = ax.imshow(img, cmap=cmap)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'step_by_step_decoding.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 4 : Impact of threshold on segmentation quality ─────────────────────
import matplotlib.pyplot as plt

THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

model.eval()
# Collect predictions on the full val set once
all_logits, all_masks = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        logits = model(imgs.to(DEVICE)).cpu()
        all_logits.append(logits)
        all_masks.append(masks)

all_logits = torch.cat(all_logits)
all_masks  = torch.cat(all_masks)

# ── Dice vs threshold curve ──
dice_per_t = []
for t in THRESHOLDS:
    preds = (torch.sigmoid(all_logits) > t).float()
    inter = (preds * all_masks).sum()
    union = preds.sum() + all_masks.sum()
    dice  = (2. * inter / (union + 1e-6)).item()
    dice_per_t.append(dice)

best_t = THRESHOLDS[dice_per_t.index(max(dice_per_t))]
print(f'Best threshold : {best_t}  →  Dice = {max(dice_per_t):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: curve
axes[0].plot(THRESHOLDS, dice_per_t, 'o-', color='steelblue', linewidth=2)
axes[0].axvline(best_t, color='red', linestyle='--', label=f'Best t={best_t}')
axes[0].axvline(THRESHOLD, color='gray', linestyle=':', label=f'Default t={THRESHOLD}')
axes[0].set_title('Val Dice vs threshold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Dice score')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: side-by-side masks at different thresholds for one sample
sample_logit = all_logits[0].unsqueeze(0)
show_t = [0.2, 0.4, 0.5, 0.6, 0.8]
axes[1].axis('off')
fig2, axes2 = plt.subplots(1, len(show_t) + 1, figsize=(3 * (len(show_t)+1), 3))
axes2[0].imshow(all_masks[0].squeeze(), cmap='gray'); axes2[0].set_title('GT'); axes2[0].axis('off')
for ax, t in zip(axes2[1:], show_t):
    pred = (torch.sigmoid(sample_logit) > t).float().squeeze()
    d    = dice_score(sample_logit, all_masks[0:1], threshold=t)
    ax.imshow(pred, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f't={t}\nDice={d:.2f}', fontsize=9)
    ax.axis('off')
fig2.suptitle('Mask at different thresholds (one sample)', fontsize=11)
fig.tight_layout(); plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'threshold_curve.png',   dpi=150, bbox_inches='tight')
fig2.savefig(OUTPUT_DIR / 'threshold_masks.png',  dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 5 : Évaluation finale complète ──────────────────────────────────────
from sklearn.metrics import confusion_matrix

model.eval()
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))

results = []

with torch.no_grad():
    for imgs, masks in val_loader:
        logits = model(imgs.to(DEVICE)).cpu()
        probs  = torch.sigmoid(logits)
        preds  = (probs > THRESHOLD).float()

        for i in range(len(imgs)):
            p = preds[i].squeeze().numpy().flatten().astype(int)
            g = masks[i].squeeze().numpy().flatten().astype(int)

            tp = int(((p == 1) & (g == 1)).sum())
            fp = int(((p == 1) & (g == 0)).sum())
            fn = int(((p == 0) & (g == 1)).sum())
            tn = int(((p == 0) & (g == 0)).sum())

            dice      = (2*tp + 1e-6) / (2*tp + fp + fn + 1e-6)
            iou       = (tp + 1e-6)   / (tp + fp + fn + 1e-6)
            precision = (tp + 1e-6)   / (tp + fp + 1e-6)
            recall    = (tp + 1e-6)   / (tp + fn + 1e-6)
            results.append(dict(dice=dice, iou=iou, precision=precision, recall=recall))

import pandas as pd
df = pd.DataFrame(results)

print('=' * 48)
print(f'  {"Metric":<12} {"Mean":>8}  {"Std":>8}  {"Min":>8}  {"Max":>8}')
print('=' * 48)
for col in ['dice', 'iou', 'precision', 'recall']:
    print(f'  {col:<12} {df[col].mean():>8.4f}  {df[col].std():>8.4f}'
          f'  {df[col].min():>8.4f}  {df[col].max():>8.4f}')
print('=' * 48)

# Histogram of per-image Dice
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Per-image metric distributions (val set)', fontsize=12, fontweight='bold')
colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid']
for ax, col, c in zip(axes, ['dice', 'iou', 'precision', 'recall'], colors):
    ax.hist(df[col], bins=30, color=c, edgecolor='white', linewidth=0.5)
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={df[col].mean():.3f}')
    ax.set_title(col.capitalize())
    ax.set_xlabel('Score')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
df.to_csv(OUTPUT_DIR / 'per_image_metrics.csv', index=False)
print(f'\nPer-image CSV saved → {OUTPUT_DIR}/per_image_metrics.csv')

In [ ]:
# ── CELL 7 : Résumé final ─────────────────────────────────────────────────────
import json, datetime

summary = {
    'date'           : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model'          : f'U-Net / {ENCODER}',
    'encoder_weights': ENCODER_WEIGHTS,
    'img_size'       : IMG_SIZE,
    'epochs_run'     : len(history['train_loss']),
    'best_val_loss'  : round(best_val_loss, 4),
    'best_val_dice'  : round(best_val_dice, 4),
    'threshold'      : THRESHOLD,
    'checkpoint'     : str(BEST_CKPT),
}

# Add final eval metrics if Cell 5 was run
try:
    summary['val_dice_mean']      = round(float(df['dice'].mean()),      4)
    summary['val_iou_mean']       = round(float(df['iou'].mean()),       4)
    summary['val_precision_mean'] = round(float(df['precision'].mean()), 4)
    summary['val_recall_mean']    = round(float(df['recall'].mean()),     4)
except NameError:
    pass  # Cell 5 wasn't run — skip

# Save
summary_path = OUTPUT_DIR / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('╔══════════════════════════════════════════════════╗')
print('║              RÉSUMÉ FINAL — ENTRAÎNEMENT         ║')
print('╠══════════════════════════════════════════════════╣')
for k, v in summary.items():
    print(f'║  {k:<25} {str(v):<22} ║')
print('╠══════════════════════════════════════════════════╣')
print('║  PROCHAINES ÉTAPES                               ║')
print('║  1. Essayer Dice + Focal (Cell 6)                ║')
print('║  2. Augmenter résolution à 640 si VRAM le permet ║')
print('║  3. TTA (Test-Time Augmentation) à l\'inférence   ║')
print('║  4. Ensemble de modèles (FPN + U-Net)            ║')
print('╚══════════════════════════════════════════════════╝')
print(f'\nSummary JSON saved → {summary_path}')

In [ ]:
def postprocess_mask(prob_map: np.ndarray,
                     threshold: float = THRESHOLD,
                     min_area:  int   = MIN_AREA) -> np.ndarray:
    """
    Converts a probability map (H,W) float32 [0,1] to a clean binary mask.

    Steps:
      1. Sigmoid already applied upstream; just threshold.
      2. Remove small connected components (noise).
      3. Fill internal holes.
      4. Morphological opening (separates fused blobs).

    Returns uint8 mask {0, 1}.
    """
    # 1. Threshold
    binary = (prob_map > threshold).astype(np.uint8)

    # 2. Remove small connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )
    cleaned = np.zeros_like(binary)
    for lbl in range(1, num_labels):          # label 0 = background
        if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == lbl] = 1

    # 3. Fill holes (flood-fill from border → invert → OR with cleaned)
    filled     = cleaned.copy()
    h, w       = filled.shape
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(filled, flood_mask, seedPoint=(0, 0), newVal=2)
    holes      = (filled == 0)     # pixels never reached by flood = holes
    filled     = np.where(holes, 1, cleaned).astype(np.uint8)

    # 4. Morphological opening (erode then dilate) — removes thin protrusions
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opened = cv2.morphologyEx(filled, cv2.MORPH_OPEN, kernel, iterations=1)

    return opened


print('Post-processing function defined.')

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model.eval()
print(f'Loaded best model from {BEST_CKPT}')


@torch.no_grad()
def predict_batch(model, images: torch.Tensor, device) -> np.ndarray:
    """
    Forward pass → sigmoid → CPU numpy array (B, H, W) float32 [0,1].
    """
    images = images.to(device, non_blocking=True)
    logits = model(images)                  # (B, 1, H, W)
    probs  = torch.sigmoid(logits)          # (B, 1, H, W)
    return probs[:, 0].cpu().numpy()        # (B, H, W)


# ── Run inference + collect Dice on test split ────────────────────────────────
test_dice_scores = []

for images, masks in test_loader:
    prob_maps = predict_batch(model, images, DEVICE)     # (B, H, W)
    mask_np   = masks[:, 0].cpu().numpy()                # (B, H, W)

    for prob, gt in zip(prob_maps, mask_np):
        pred = postprocess_mask(prob)
        inter = (pred * gt).sum()
        union = pred.sum() + gt.sum()
        d     = (2 * inter + 1e-6) / (union + 1e-6)
        test_dice_scores.append(d)

mean_dice = np.mean(test_dice_scores)
std_dice  = np.std(test_dice_scores)
print(f'\nTest Dice  :  mean = {mean_dice:.4f}  |  std = {std_dice:.4f}')
print(f'Min        :  {min(test_dice_scores):.4f}')
print(f'Max        :  {max(test_dice_scores):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(test_dice_scores, bins=30, color='#185FA5', edgecolor='white', linewidth=0.5)
ax.axvline(mean_dice, color='#D85A30', lw=2, label=f'Mean = {mean_dice:.3f}')
ax.axvline(np.median(test_dice_scores), color='#3B6D11', lw=1.5, ls='--',
           label=f'Median = {np.median(test_dice_scores):.3f}')

ax.set_xlabel('Dice score', fontsize=11)
ax.set_ylabel('Count',      fontsize=11)
ax.set_title('Dice score distribution — test set', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dice_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Save full model (architecture + weights) for portability
FINAL_CKPT = OUTPUT_DIR / 'pipeline1_final.pth'
torch.save({
    'model_state_dict' : model.state_dict(),
    'encoder'          : ENCODER,
    'encoder_weights'  : ENCODER_WEIGHTS,
    'img_size'         : IMG_SIZE,
    'threshold'        : THRESHOLD,
    'best_val_dice'    : best_val_dice,
    'test_dice_mean'   : mean_dice,
    'test_dice_std'    : std_dice,
}, FINAL_CKPT)

print('═' * 55)
print('  Pipeline 1 — Final Summary')
print('═' * 55)
print(f'  Encoder         : {ENCODER} (pretrained={ENCODER_WEIGHTS})')
print(f'  Decoder         : U-Net')
print(f'  Image size      : {IMG_SIZE}×{IMG_SIZE}')
print(f'  Loss            : {DICE_WEIGHT}×Dice + {BCE_WEIGHT}×FL')
print(f'  Best val loss   : {best_val_loss:.4f}')
print(f'  Best val Dice   : {best_val_dice:.4f}')
print(f'  Test Dice       : {mean_dice:.4f} ± {std_dice:.4f}')
print(f'  Checkpoint      : {FINAL_CKPT}')
print('═' * 55)

# testing segmentation model on 4 grades dataset

In [ ]:
!pip install -q --no-deps segmentation_models_pytorch ttach
!pip install -q albumentations

In [ ]:
import time
from pathlib import Path


import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau


from segmentation_models_pytorch.losses import DiceLoss

In [ ]:
import os, glob, random
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Path to the new inspection dataset
INSPECT_ROOT = Path('/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes/train')  # adjust if needed

GRADE_DIRS = {
    'Grade 1': INSPECT_ROOT / 'Grade 1',
    'Grade 2': INSPECT_ROOT / 'Grade 2',
    'Grade 3': INSPECT_ROOT / 'Grade 3',
    'Grade 4': INSPECT_ROOT / 'Grade 4',
}

N_PER_CLASS = 6   # 6 * 4 = 24 samples total (tweak to land in your 20-30 range)
SEED_INSPECT = 42

for name, d in GRADE_DIRS.items():
    print(name, '->', d, '| exists:', d.exists())

In [ ]:
random.seed(SEED_INSPECT)

sample_paths = []   # list of (grade_label, image_path)

for grade_name, grade_dir in GRADE_DIRS.items():
    files = sorted(glob.glob(os.path.join(str(grade_dir), '*.jpg'))) + \
            sorted(glob.glob(os.path.join(str(grade_dir), '*.png')))
    if len(files) == 0:
        print(f'⚠️  No images found in {grade_dir}')
        continue
    chosen = random.sample(files, min(N_PER_CLASS, len(files)))
    sample_paths.extend([(grade_name, f) for f in chosen])

print(f'Total samples selected: {len(sample_paths)}')
for g, p in sample_paths[:5]:
    print(g, '->', p)

In [ ]:
inference_model = smp.Unet(
    encoder_name    = 'resnet34',           # 'resnet34'
    encoder_weights = None,              # weights come from checkpoint, not imagenet re-download
    in_channels     = 3,
    classes         = 1,
    activation      = None,
)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BEST_CKPT = '/kaggle/input/models/saharfeki/unet-aug-0-82/pytorch/default/1/0.82 model with augmentation.pth'   # reuse from training config
state_dict = torch.load(BEST_CKPT, map_location=DEVICE)
inference_model.load_state_dict(state_dict)
inference_model = inference_model.to(DEVICE)
inference_model.eval()

print(f'Loaded checkpoint: {BEST_CKPT}')

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

infer_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

@torch.no_grad()
def predict_mask(image_path, model, device, threshold=0.5, min_area=500):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise FileNotFoundError(f'Cannot load image: {image_path}')
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = image_rgb.shape[:2]

    aug = infer_transform(image=image_rgb)
    input_tensor = aug['image'].unsqueeze(0).to(device)  # (1, C, H, W)

    logits = model(input_tensor)
    probs  = torch.sigmoid(logits)[0, 0].cpu().numpy()   # (IMG_SIZE, IMG_SIZE)

    # Threshold
    pred_mask = (probs >= threshold).astype(np.uint8)

    # Remove small connected components
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_mask, connectivity=8)
    clean_mask = np.zeros_like(pred_mask)
    for lbl in range(1, n_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
            clean_mask[labels == lbl] = 1

    # Resize mask back to original image size for overlay
    clean_mask_full = cv2.resize(clean_mask.astype(np.uint8), (orig_w, orig_h),
                                  interpolation=cv2.INTER_NEAREST)

    return image_rgb, clean_mask_full, probs

In [ ]:
results = []  # (grade_label, image_rgb, pred_mask, wound_pixel_ratio)

for grade_name, img_path in sample_paths:
    image_rgb, pred_mask, probs = predict_mask(img_path, inference_model, DEVICE)
    wound_ratio = pred_mask.sum() / pred_mask.size
    results.append((grade_name, os.path.basename(img_path), image_rgb, pred_mask, wound_ratio))

print(f'Predictions done on {len(results)} images.')
for g, name, _, _, ratio in results[:8]:
    print(f'{g:10s} | {name:30s} | wound area: {ratio*100:.2f}%')

In [ ]:
OUTPUT_DIR  = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def make_overlay(image_rgb, mask, color=(255, 0, 0), alpha=0.45):
    overlay = image_rgb.copy()
    colored = np.zeros_like(image_rgb)
    colored[mask == 1] = color
    return cv2.addWeighted(overlay, 1 - alpha, colored, alpha, 0)

n_cols = 4
n_rows = int(np.ceil(len(results) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
axes = axes.flatten()

for ax, (grade_name, fname, image_rgb, pred_mask, ratio) in zip(axes, results):
    overlay = make_overlay(image_rgb, pred_mask)
    ax.imshow(overlay)
    ax.set_title(f'{grade_name}\n{fname}\narea: {ratio*100:.1f}%', fontsize=8)
    ax.axis('off')

# Hide unused axes
for ax in axes[len(results):]:
    ax.axis('off')

plt.tight_layout()
save_path = OUTPUT_DIR / 'inspection_grid_4classes.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved grid to {save_path}')

In [ ]:
# ── List all photos used in inspection_grid_4classes.png ───────────────────
print(f'Photos used in inspection_grid_4classes.png ({len(results)} total):\n')
n_cols = 4
n_rows = int(np.ceil(len(results) / n_cols))
for i, (grade_name, fname, _, _, ratio) in enumerate(results, start=1):
    row = (i - 1) // n_cols + 1
    col = (i - 1) % n_cols + 1
    print(f'{i:2d}. [row {row}, col {col}]  {grade_name:10s}  {fname:30s}  wound area: {ratio*100:.2f}%')

# Optional: also save this list to a text file alongside the grid image
names_txt_path = OUTPUT_DIR / 'inspection_grid_4classes_filenames.txt'
with open(names_txt_path, 'w') as f:
    for i, (grade_name, fname, _, _, ratio) in enumerate(results, start=1):
        f.write(f'{i:2d}. {grade_name:10s}  {fname:30s}  wound area: {ratio*100:.2f}%\n')

print(f'\nSaved list → {names_txt_path}')

# per-grade Dice breakdown

In [ ]:
!pip install -q --no-deps segmentation_models_pytorch ttach
!pip install -q albumentations

import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
BG_MODEL_PATH  = "/kaggle/input/models/saharfeki/unet-bg-removal/pytorch/default/1/best_model bg_remove.pth"
AUG_MODEL_PATH = "/kaggle/input/models/saharfeki/unet-aug-0-82/pytorch/default/1/0.82 model with augmentation.pth"

DFUC_IMG_DIR  = "/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_images"
DFUC_MASK_DIR = "/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_masks"

GRADE_ROOT = "/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes/train"
GRADE_DIRS = {
    1: os.path.join(GRADE_ROOT, "Grade 1"),
    2: os.path.join(GRADE_ROOT, "Grade 2"),
    3: os.path.join(GRADE_ROOT, "Grade 3"),
    4: os.path.join(GRADE_ROOT, "Grade 4"),
}
for g, p in GRADE_DIRS.items():
    print(g, os.path.exists(p), len(os.listdir(p)) if os.path.exists(p) else "MISSING")

In [ ]:
IMG_SIZE = 512  # EDIT to match your training resolution

def build_model():
    model = smp.Unet(

        encoder_name="resnet34",
        encoder_weights=None,   # weights come from checkpoint, not ImageNet, at load time
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model

def load_checkpoint(model, path, device):
    ckpt = torch.load(path, map_location=device)
    # handle both raw state_dict and dict-wrapped checkpoints
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt
    # strip "module." prefix if saved from DataParallel
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()
    return model

model_bg  = load_checkpoint(build_model(), BG_MODEL_PATH, device)
model_aug = load_checkpoint(build_model(), AUG_MODEL_PATH, device)
print("Both models loaded.")

In [ ]:
# EDIT mean/std if you used something other than ImageNet stats during training
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def get_transform(bg_removal_fn=None):
    """bg_removal_fn: pass your apply_background_removal function for the bg model, None for aug model"""
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

transform_plain = get_transform()



def min_max_normalize(image_rgb):
    img = image_rgb.astype(np.float32)
    img_min, img_max = img.min(), img.max()
    if img_max - img_min < 1e-6:
        return image_rgb.astype(np.uint8)
    norm = (img - img_min) / (img_max - img_min)
    return (norm * 255).astype(np.uint8)


def fill_holes_flood(mask):
    """Border flood-fill: fills only holes fully unreachable from the image edge."""
    h, w = mask.shape
    flood_fill_mask = np.zeros((h + 2, w + 2), np.uint8)
    mask_copy = mask.copy()
    cv2.floodFill(mask_copy, flood_fill_mask, (0, 0), 255)
    background_reachable_inv = cv2.bitwise_not(mask_copy)
    filled = cv2.bitwise_or(mask, background_reachable_inv)
    return filled


def fill_via_convex_hull(mask):
    """
    Fills concave bays/notches that touch the border or connect to true
    background through a thin channel (which fill_holes_flood cannot fix,
    since it only fills fully-enclosed holes).
    Replaces each connected component with its convex hull — guarantees no
    notch survives regardless of how it connects to the outside background.
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull_mask = np.zeros_like(mask)
    for cnt in contours:
        if cv2.contourArea(cnt) < 50:   # skip tiny noise contours
            continue
        hull = cv2.convexHull(cnt)
        cv2.drawContours(hull_mask, [hull], -1, 255, thickness=cv2.FILLED)
    return hull_mask



def keep_components_above_area(mask, min_area_frac=0.02):
    h, w = mask.shape
    min_area = min_area_frac * h * w
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == label] = 255
    return cleaned


def get_skin_mask(image_rgb, median_kernel=15, min_area_frac=0.02, closing_frac=0.05):
    """
    Background removal via color-space thresholding.
    - Min-max normalize image intensities first
    - Otsu threshold on Cr (YCbCr) and a* (CIELAB), combined with OR
    - Median filter + morphological closing (kernel scaled to image size)
    - Border flood-fill for fully-enclosed gaps
    - Convex-hull fill per component to eliminate any remaining bay/notch,
      including ones connected to background through the image border
      (fixes wound-bed cavities being carved out as "background")
    - Area-threshold component filtering to keep all significant blobs
    """
    h, w = image_rgb.shape[:2]
    norm_img = min_max_normalize(image_rgb)

    ycrcb = cv2.cvtColor(norm_img, cv2.COLOR_RGB2YCrCb)
    cr = ycrcb[:, :, 1]
    _, cr_mask = cv2.threshold(cr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    lab = cv2.cvtColor(norm_img, cv2.COLOR_RGB2LAB)
    a_ch = lab[:, :, 1]
    _, a_mask = cv2.threshold(a_ch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    combined = cv2.bitwise_or(cr_mask, a_mask)

    k = int(min(h, w) * 0.02) | 1   # odd kernel, scaled to image size
    k = max(k, median_kernel)
    cleaned = cv2.medianBlur(combined, k)

    # Closing kernel scaled to image size — big enough to bridge large notches
    closing_kernel = max(21, int(min(h, w) * closing_frac)) | 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (closing_kernel, closing_kernel))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)

    cleaned = fill_holes_flood(cleaned)
    cleaned = keep_components_above_area(cleaned, min_area_frac)
    cleaned = fill_via_convex_hull(cleaned)          # NEW — guarantees no notch survives

    return cleaned


def apply_background_removal(image_rgb, median_kernel=15, min_area_frac=0.02, closing_frac=0.05):
    mask = get_skin_mask(image_rgb, median_kernel=median_kernel,
                          min_area_frac=min_area_frac, closing_frac=closing_frac)
    mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB) // 255
    masked = (image_rgb * mask_3ch).astype(np.uint8)
    return masked, mask

In [ ]:
# ── Cell 6 (final): Rebuild the exact same seeded split + two test datasets ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
class WoundDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, remove_background=False):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform
        self.remove_background = remove_background

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(str(self.image_paths[idx]))
        if image is None:
            raise FileNotFoundError(f'Cannot load image: {self.image_paths[idx]}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.remove_background:
            image, _ = apply_background_removal(image)

        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f'Cannot load mask: {self.mask_paths[idx]}')
        mask = (mask >= 128).astype(np.float32)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask']

        mask = mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.tensor(mask).unsqueeze(0)
        fname = os.path.basename(str(self.image_paths[idx]))
        return image, mask, fname


# ── Exact split params from training notebook ───────────────────────────────
SEED        = 42
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10

import glob as glob_module

IMAGE_DIR = DFUC_IMG_DIR
MASK_DIR  = DFUC_MASK_DIR

image_paths = sorted(glob_module.glob(os.path.join(IMAGE_DIR, '*.jpg')))
mask_paths  = sorted(glob_module.glob(os.path.join(MASK_DIR, '*.png')))
print(f"Images found: {len(image_paths)}")
print(f"Masks found:  {len(mask_paths)}")

N = len(mask_paths)
assert len(image_paths) == N, "Image/mask count mismatch — check filename pairing before proceeding"

rng     = np.random.default_rng(SEED)
indices = rng.permutation(N)
n_train = int(N * TRAIN_RATIO)
n_val   = int(N * VAL_RATIO)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

test_images = [image_paths[i] for i in test_idx]
test_masks  = [mask_paths[i]  for i in test_idx]

print(f'Test set: {len(test_images)} images ({len(test_images)/N*100:.0f}%)')

# ── Two test datasets, same images, different preprocessing per model ───────
test_dataset_bg  = WoundDataset(test_images, test_masks, transform=val_transform, remove_background=True)
test_dataset_aug = WoundDataset(test_images, test_masks, transform=val_transform, remove_background=False)

print(f"test_dataset_bg  ready: {len(test_dataset_bg)} samples (background removed)")
print(f"test_dataset_aug ready: {len(test_dataset_aug)} samples (no background removal)")

In [ ]:
# ── Cell 7 (corrected): Dice sanity check on DFUC2022 test set ─────────────
BATCH_SIZE      = 8
NUM_WORKERS     = 0
def dice_score(pred, gt, eps=1e-7):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    return (2 * inter + eps) / (pred.sum() + gt.sum() + eps)

@torch.no_grad()
def run_dice_eval(model, dataset, device, threshold=0.5):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    scores = []
    for images, masks, fnames in tqdm(loader):
        images = images.to(device)
        probs = torch.sigmoid(model(images)).cpu().numpy()
        preds = (probs > threshold).astype(np.float32)
        masks_np = masks.numpy()
        for i in range(images.shape[0]):
            d = dice_score(preds[i, 0], masks_np[i, 0])
            scores.append({"filename": fnames[i], "dice": d})
    return pd.DataFrame(scores)

df_dice_bg  = run_dice_eval(model_bg,  test_dataset_bg,  device)
df_dice_aug = run_dice_eval(model_aug, test_dataset_aug, device)

print(f"BG-removal model  — Dice: {df_dice_bg['dice'].mean():.4f} ± {df_dice_bg['dice'].std():.4f}  (reported: 0.7138 ± 0.3121)")
print(f"Aug-only model    — Dice: {df_dice_aug['dice'].mean():.4f} ± {df_dice_aug['dice'].std():.4f}  (reported: 0.7018 ± 0.3159)")

In [ ]:
# ── Cell 8 (fixed): Grade dataset (no masks) — inference-only ──────────────

class GradeDataset(Dataset):
    def __init__(self, grade_dirs, transform, use_bg_removal=False):
        self.samples = []
        for grade, folder in grade_dirs.items():
            for fname in os.listdir(folder):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(folder, fname), grade, fname))
        self.transform = transform
        self.use_bg_removal = use_bg_removal

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, grade, fname = self.samples[idx]
        image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        orig_shape = image.shape[:2]

        if self.use_bg_removal:
            image, _ = apply_background_removal(image)   # FIXED: unpack tuple

        augmented = self.transform(image=image)
        return augmented["image"], grade, fname, orig_shape

grade_ds_bg  = GradeDataset(GRADE_DIRS, transform_plain, use_bg_removal=True)
grade_ds_aug = GradeDataset(GRADE_DIRS, transform_plain, use_bg_removal=False)
print(f"{len(grade_ds_bg)} grade images found")

In [ ]:
@torch.no_grad()
def run_inference(model, dataset, device, threshold=0.5):
    loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2)
    records = []
    all_preds = {}  # fname -> binary mask (resized to IMG_SIZE)
    for images, grades, fnames, orig_shapes in tqdm(loader):
        images = images.to(device)
        probs = torch.sigmoid(model(images)).cpu().numpy()
        preds = (probs > threshold).astype(np.float32)
        for i in range(images.shape[0]):
            pred = preds[i, 0]
            area_pct = 100 * pred.sum() / pred.size
            records.append({
                "filename": fnames[i],
                "grade": int(grades[i]),
                "pred_area_pct": area_pct,
                "pred_empty": pred.sum() == 0,
            })
            all_preds[fnames[i]] = pred
    return pd.DataFrame(records), all_preds

df_grade_bg, preds_bg   = run_inference(model_bg,  grade_ds_bg,  device)
df_grade_aug, preds_aug = run_inference(model_aug, grade_ds_aug, device)

In [ ]:
def summarize(df, label):
    s = df.groupby("grade").agg(
        n=("pred_area_pct", "count"),
        mean_area_pct=("pred_area_pct", "mean"),
        std_area_pct=("pred_area_pct", "std"),
        pct_empty=("pred_empty", "mean"),
    ).round(3)
    s.columns = [f"{c}_{label}" for c in s.columns]
    return s

summary_bg  = summarize(df_grade_bg, "bg")
summary_aug = summarize(df_grade_aug, "aug")
summary = summary_bg.join(summary_aug)
print(summary)

# Inter-model agreement: treat aug model's mask as pseudo-GT for bg model and vice versa.
# Low agreement = the two models disagree on where the wound is -> flag for manual review.
agreement_records = []
common_files = set(preds_bg.keys()) & set(preds_aug.keys())
grade_lookup = dict(zip(df_grade_bg["filename"], df_grade_bg["grade"]))

for fname in common_files:
    d = dice_score(preds_bg[fname], preds_aug[fname])
    agreement_records.append({"filename": fname, "grade": grade_lookup[fname], "agreement_dice": d})

df_agreement = pd.DataFrame(agreement_records)
agreement_summary = df_agreement.groupby("grade")["agreement_dice"].agg(["mean", "std", "min"]).round(3)
print("\nInter-model agreement (bg vs aug) by grade:")
print(agreement_summary)

In [ ]:
from pathlib import Path
OUTPUT_DIR  = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def show_disagreements(grade, n=4, save_path=None):
    subset = df_agreement[df_agreement["grade"] == grade].nsmallest(n, "agreement_dice")

    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))

    # If n=1, axes isn't iterable
    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, subset.iterrows()):
        fname = row["filename"]
        path = [p for p, g, f in grade_ds_bg.samples if f == fname][0]

        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        overlay = img_resized.copy()
        overlay[preds_bg[fname] > 0] = [255, 0, 0]      # red = bg model
        overlay[preds_aug[fname] > 0] = [0, 255, 0]     # green = aug model

        ax.imshow(cv2.addWeighted(img_resized, 0.5, overlay, 0.5, 0))
        ax.set_title(f"{fname}\nagreement: {row['agreement_dice']:.2f}", fontsize=9)
        ax.axis("off")

    plt.suptitle(
        f"Grade {grade} — worst model agreement (red=bg, green=aug, yellow=overlap)"
    )
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

for g in [1, 2, 3, 4]:
    save_path = OUTPUT_DIR / f"strong disagreement between model_grade{g}.png"
    show_disagreements(g, save_path=save_path)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
combined = pd.concat([
    df_grade_bg.assign(model="bg_removed"),
    df_grade_aug.assign(model="aug_only"),
])
sns.boxplot(data=combined, x="grade", y="pred_area_pct", hue="model", ax=axes[0])
axes[0].set_title("Predicted wound area % by grade")

empty_rates = combined.groupby(["grade", "model"])["pred_empty"].mean().reset_index()
sns.barplot(data=empty_rates, x="grade", y="pred_empty", hue="model", ax=axes[1])
axes[1].set_title("Empty-mask prediction rate by grade")
axes[1].set_ylabel("fraction predicted empty")

plt.tight_layout()
save_path = OUTPUT_DIR / "comparaison stat between model.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

# Stage classification 

In [ ]:
!pip install -q --no-deps segmentation_models_pytorch ttach
!pip install -q albumentations

import os, glob, random, json
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import segmentation_models_pytorch as smp
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

In [ ]:
# Adjust these to your actual Kaggle input paths
BASE_DIR = '/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes'
GRADE_DIRS = {
    'train': {
        1: f'{BASE_DIR}/train/Grade 1', 2: f'{BASE_DIR}/train/Grade 2',
        3: f'{BASE_DIR}/train/Grade 3', 4: f'{BASE_DIR}/train/Grade 4',
    },
    'val': {
        1: f'{BASE_DIR}/val/Grade 1', 2: f'{BASE_DIR}/val/Grade 2',
        3: f'{BASE_DIR}/val/Grade 3', 4: f'{BASE_DIR}/val/Grade 4',
    },
    'test': {
        1: f'{BASE_DIR}/test/Grade 1', 2: f'{BASE_DIR}/test/Grade 2',
        3: f'{BASE_DIR}/test/Grade 3', 4: f'{BASE_DIR}/test/Grade 4',
    },
}

UNET_WEIGHTS = '/kaggle/input/models/saharfeki/unet-aug-0-82/pytorch/default/1/0.82 model with augmentation.pth'
CROP_OUTPUT_DIR = '/kaggle/working/crops'   # cached crops go here
IMG_SIZE = 224
NUM_CLASSES = 4
PAD_RATIO = 0.15   # bbox padding margin

os.makedirs(CROP_OUTPUT_DIR, exist_ok=True)

In [ ]:
unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,   # weights come from your checkpoint, not ImageNet, at inference time
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

checkpoint = torch.load(UNET_WEIGHTS, map_location=DEVICE)
# Handle both raw state_dict and wrapped checkpoint formats
state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
unet_model.load_state_dict(state_dict)
unet_model.eval()
print("U-Net loaded.")

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_and_preprocess(img_path, size=256):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig = img.copy()
    img_resized = cv2.resize(img, (size, size))
    img_norm = img_resized.astype(np.float32) / 255.0
    img_norm = (img_norm - IMAGENET_MEAN) / IMAGENET_STD   # <-- add this
    tensor = torch.from_numpy(img_norm.transpose(2,0,1)).unsqueeze(0).float()
    return tensor, orig

def predict_mask(img_path, threshold=0.5, size=256):
    tensor, orig = load_and_preprocess(img_path, size)
    with torch.no_grad():
        pred = torch.sigmoid(unet_model(tensor.to(DEVICE)))
    mask = pred.squeeze().cpu().numpy()
    mask_bin = (mask > threshold).astype(np.uint8)
    mask_bin = cv2.resize(mask_bin, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask_bin, orig

# Visually inspect ~6 random samples across grades before trusting this at scale
sample_paths = []

for grade, folder in GRADE_DIRS['train'].items():
    files = glob.glob(f"{folder}/*.jpg") + glob.glob(f"{folder}/*.png")
    selected = random.sample(files, min(2, len(files)))

    # Store (image_path, grade)
    sample_paths.extend([(f, grade) for f in selected])

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(10, 4 * len(sample_paths)))

# Handle the case where there is only one sample
if len(sample_paths) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, (p, grade) in enumerate(sample_paths):
    mask, orig = predict_mask(p)

    # Create overlay
    overlay = orig.copy()
    overlay[mask == 1] = [255, 0, 0]   # Red in RGB
    blended = cv2.addWeighted(orig, 0.7, overlay, 0.3, 0)

    # Original image
    axes[i, 0].imshow(orig)
    axes[i,0].set_title(f"Original Image ({grade})")
    axes[i, 0].axis("off")

    # Overlay image
    axes[i, 1].imshow(blended)

    if np.sum(mask) == 0:
        axes[i, 1].set_title("Predicted Mask Overlay\n(No mask predicted)",
                             color="red",
                             fontsize=11)
        axes[i, 1].text(
            0.5, 0.5,
            "NO MASK\nPREDICTED",
            color="red",
            fontsize=16,
            fontweight="bold",
            ha="center",
            va="center",
            transform=axes[i, 1].transAxes,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
        )
    else:
        axes[i, 1].set_title("Predicted Mask Overlay")

    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_and_preprocess(img_path, size=256):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig = img.copy()
    img_resized = cv2.resize(img, (size, size))
    img_norm = img_resized.astype(np.float32) / 255.0
    img_norm = (img_norm - IMAGENET_MEAN) / IMAGENET_STD   # <-- add this
    tensor = torch.from_numpy(img_norm.transpose(2,0,1)).unsqueeze(0).float()
    return tensor, orig

def predict_mask(img_path, threshold=0.5, size=256):
    tensor, orig = load_and_preprocess(img_path, size)
    with torch.no_grad():
        pred = torch.sigmoid(unet_model(tensor.to(DEVICE)))
    mask = pred.squeeze().cpu().numpy()
    mask_bin = (mask > threshold).astype(np.uint8)
    mask_bin = cv2.resize(mask_bin, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask_bin, orig
sample_dir = "/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_images"

all_images = glob.glob(f"{sample_dir}/*.jpg") + \
             glob.glob(f"{sample_dir}/*.png") + \
             glob.glob(f"{sample_dir}/*.jpeg")

sample_paths = random.sample(all_images, min(6, len(all_images)))
fig, axes = plt.subplots(len(sample_paths), 2, figsize=(10, 4 * len(sample_paths)))

# Handle the case where there is only one sample
if len(sample_paths) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, p in enumerate(sample_paths):
    mask, orig = predict_mask(p)

    # Create overlay
    overlay = orig.copy()
    overlay[mask == 1] = [255, 0, 0]   # Red in RGB
    blended = cv2.addWeighted(orig, 0.7, overlay, 0.3, 0)

    # Original image
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title("Original Image")
    axes[i, 0].axis("off")

    # Overlay image
    axes[i, 1].imshow(blended)

    if np.sum(mask) == 0:
        axes[i, 1].set_title("Predicted Mask Overlay\n(No mask predicted)",
                             color="red",
                             fontsize=11)
        axes[i, 1].text(
            0.5, 0.5,
            "NO MASK\nPREDICTED",
            color="red",
            fontsize=16,
            fontweight="bold",
            ha="center",
            va="center",
            transform=axes[i, 1].transAxes,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
        )
    else:
        axes[i, 1].set_title("Predicted Mask Overlay")

    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

# nnunet

In [ ]:
!pip install nnunetv2 -q

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import os

nnUNet_raw = "/kaggle/working/nnUNet_raw"
nnUNet_preprocessed = "/kaggle/working/nnUNet_preprocessed"
nnUNet_results = "/kaggle/working/nnUNet_results"

for p in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(p, exist_ok=True)

os.environ["nnUNet_raw"] = nnUNet_raw
os.environ["nnUNet_preprocessed"] = nnUNet_preprocessed
os.environ["nnUNet_results"] = nnUNet_results

print(os.environ["nnUNet_raw"])

In [ ]:
import os
import shutil
import numpy as np
from PIL import Image
from glob import glob

dataset_dir = "/kaggle/working/nnUNet_raw/Dataset001_DFU"
for sub in ["imagesTr", "labelsTr", "imagesTs"]:
    os.makedirs(os.path.join(dataset_dir, sub), exist_ok=True)

src_images = sorted(glob("/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_images/*.jpg"))
src_masks  = sorted(glob("/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_masks/*.png"))

assert len(src_images) == len(src_masks), f"{len(src_images)} images vs {len(src_masks)} masks"

for img_path, mask_path in zip(src_images, src_masks):
    case_id = os.path.splitext(os.path.basename(img_path))[0]

    # --- image: jpg -> png, keep RGB ---
    img = Image.open(img_path).convert("RGB")
    img.save(os.path.join(dataset_dir, "imagesTr", f"{case_id}_0000.png"))

    # --- mask: convert to single-channel binary {0,1} ---
    mask = Image.open(mask_path).convert("L")   # collapse to grayscale if it's RGB
    mask_arr = np.array(mask)
    binary_arr = (mask_arr > 127).astype(np.uint8)  # threshold handles anti-aliasing/stray values
    Image.fromarray(binary_arr, mode="L").save(os.path.join(dataset_dir, "labelsTr", f"{case_id}.png"))

print("Done. Converted", len(src_images), "pairs.")

In [ ]:
img_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in src_images}
mask_dict = {os.path.splitext(os.path.basename(p))[0].replace("_mask", ""): p for p in src_masks}
common_ids = sorted(set(img_dict) & set(mask_dict))
print(f"{len(common_ids)} matched pairs out of {len(img_dict)} images / {len(mask_dict)} masks")

In [ ]:
import json

dataset_json = {
    "channel_names": {"0": "R", "1": "G", "2": "B"},
    "labels": {
        "background": 0,
        "wound": 1
    },
    "numTraining": len(common_ids),   # use the matched-pair count, not raw glob length
    "file_ending": ".png"
}

with open(os.path.join(dataset_dir, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=4)

In [ ]:
import os, shutil, random

random.seed(42)

images_dir = "/kaggle/working/nnUNet_raw/Dataset001_DFU/imagesTr"
labels_dir = "/kaggle/working/nnUNet_raw/Dataset001_DFU/labelsTr"

all_ids = sorted([f.replace("_0000.png", "") for f in os.listdir(images_dir)])
random.shuffle(all_ids)

n_test = int(0.15 * len(all_ids))  # ~15% held out, adjust as you like
test_ids = set(all_ids[:n_test])

test_images_dir = "/kaggle/working/nnUNet_raw/Dataset001_DFU/imagesTs"
test_labels_dir = "/kaggle/working/test_labels_holdout"  # keep OUTSIDE nnUNet_raw structure
os.makedirs(test_labels_dir, exist_ok=True)

for case_id in test_ids:
    shutil.move(os.path.join(images_dir, f"{case_id}_0000.png"),
                os.path.join(test_images_dir, f"{case_id}_0000.png"))
    shutil.move(os.path.join(labels_dir, f"{case_id}.png"),
                os.path.join(test_labels_dir, f"{case_id}.png"))

print(f"Held out {len(test_ids)} cases as test set, {len(all_ids)-len(test_ids)} remain for train/val")

In [ ]:
import json

json_path = "/kaggle/working/nnUNet_raw/Dataset001_DFU/dataset.json"
with open(json_path) as f:
    dj = json.load(f)
dj["numTraining"] = len(all_ids) - len(test_ids)
with open(json_path, "w") as f:
    json.dump(dj, f, indent=4)

In [ ]:
!nnUNetv2_plan_and_preprocess -d 001 --verify_dataset_integrity

In [ ]:
!nnUNetv2_train 001 2d 0 --npz

In [ ]:
!nnUNetv2_train 001 2d 0 --npz -tr nnUNetTrainer_250epochs

In [ ]:
import shutil

fold0_dir = "/kaggle/working/nnUNet_results/Dataset001_DFU/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0"
shutil.copy(
    os.path.join(fold0_dir, "checkpoint_best.pth"),
    os.path.join(fold0_dir, "checkpoint_latest.pth")
)
print("checkpoint_latest.pth created from checkpoint_best.pth")

In [ ]:
!nnUNetv2_train 001 2d 0 --npz -tr nnUNetTrainer_250epochs --c

In [ ]:
import shutil

fold0_dir = "/kaggle/working/nnUNet_results/Dataset001_DFU/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0"
shutil.copy(
    os.path.join(fold0_dir, "checkpoint_best.pth"),
    os.path.join(fold0_dir, "checkpoint_latest.pth")
)
print("checkpoint_latest.pth created from checkpoint_best.pth")

In [ ]:
from IPython.display import FileLink

FileLink("nnUNet_results/Dataset001_DFU/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0/checkpoint_latest.pth")

In [ ]:
!nnUNetv2_train 001 2d 0 --npz -tr nnUNetTrainer_250epochs --c

In [ ]:
import shutil

fold0_dir = "/kaggle/working/nnUNet_results/Dataset001_DFU/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0"
shutil.copy(
    os.path.join(fold0_dir, "checkpoint_best.pth"),
    os.path.join(fold0_dir, "checkpoint_latest.pth")
)
print("checkpoint_latest.pth created from checkpoint_best.pth")

In [ ]:
import torch
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_250epochs
from nnunetv2.utilities.dataset_name_id_conversion import maybe_convert_to_dataset_name
from nnunetv2.paths import nnUNet_preprocessed, nnUNet_results
import os
import json

dataset_name = maybe_convert_to_dataset_name(1)

plans_file = os.path.join(nnUNet_preprocessed, dataset_name, "nnUNetPlans.json")
dataset_json_file = os.path.join(nnUNet_preprocessed, dataset_name, "dataset.json")

with open(plans_file) as f:
    plans = json.load(f)
plans["continue_training"] = False   # <-- the missing key

with open(dataset_json_file) as f:
    dataset_json = json.load(f)

trainer = nnUNetTrainer_250epochs(
    plans=plans,
    configuration="2d",
    fold=0,
    dataset_json=dataset_json,
    device=torch.device("cuda")
)

trainer.initialize()

checkpoint_path = os.path.join(
    nnUNet_results, dataset_name,
    "nnUNetTrainer_250epochs__nnUNetPlans__2d", "fold_0", "checkpoint_best.pth"
)
trainer.load_checkpoint(checkpoint_path)

trainer.perform_actual_validation(save_probabilities=True)

In [ ]:
summary_path = os.path.join(
    nnUNet_results, dataset_name,
    "nnUNetTrainer_250epochs__nnUNetPlans__2d", "fold_0", "validation", "summary.json"
)
with open(summary_path) as f:
    val_summary = json.load(f)

print("Mean foreground Dice:", val_summary["foreground_mean"]["Dice"])

In [ ]:
import json
with open("/kaggle/working/nnUNet_results/Dataset001_DFU/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0/validation/summary.json") as f:
    val_summary = json.load(f)

print("Mean foreground Dice:", val_summary["foreground_mean"]["Dice"]) 

In [ ]:
!nnUNetv2_predict -i /kaggle/working/nnUNet_raw/Dataset001_DFU/imagesTs \
                   -o /kaggle/working/predictions_test \
                   -d 001 -c 2d -f 0 \
                   -tr nnUNetTrainer_250epochs \
                   -chk checkpoint_best.pth

In [ ]:
import numpy as np
from PIL import Image
import os

def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    return 2. * intersection / (pred.sum() + gt.sum() + 1e-8)

pred_dir = "/kaggle/working/predictions_test"
gt_dir = "/kaggle/working/test_labels_holdout"

dice_scores = []
for fname in os.listdir(pred_dir):
    if not fname.endswith(".png"):
        continue
    pred = np.array(Image.open(os.path.join(pred_dir, fname)))
    gt = np.array(Image.open(os.path.join(gt_dir, fname)))
    dice_scores.append(dice_score(pred, gt))

print(f"Test set mean Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")

In [ ]:
# find your worst-performing test cases to inspect
import numpy as np

scored = list(zip(os.listdir(pred_dir), dice_scores))
scored.sort(key=lambda x: x[1])
print("Worst 5 cases:", scored[:5])
print("Best 5 cases:", scored[-5:])

In [ ]:
import numpy as np
from PIL import Image
import os

worst_cases = ['101144.png', '100253.png', '101938.png', '101580.png', '100011.png']

for fname in worst_cases:
    gt = np.array(Image.open(os.path.join(gt_dir, fname)))
    pred = np.array(Image.open(os.path.join(pred_dir, fname)))
    print(f"{fname}: GT sum={gt.sum()}, Pred sum={pred.sum()}")

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os, random

images_dir = "/kaggle/working/nnUNet_raw/Dataset001_DFU/imagesTs"
gt_dir = "/kaggle/working/test_labels_holdout"
pred_dir = "/kaggle/working/predictions_test"

# get case IDs that exist in all three folders
image_files = os.listdir(images_dir)
case_ids = [f.replace("_0000.png", "") for f in image_files if f.endswith("_0000.png")]

pred_ids = set(f.replace(".png", "") for f in os.listdir(pred_dir) if f.endswith(".png"))
gt_ids = set(f.replace(".png", "") for f in os.listdir(gt_dir) if f.endswith(".png"))

valid_ids = [c for c in case_ids if c in pred_ids and c in gt_ids]
print(f"{len(valid_ids)} cases available for visualization")

n_samples = 4
sample_ids = random.sample(valid_ids, min(n_samples, len(valid_ids)))

fig, axes = plt.subplots(len(sample_ids), 3, figsize=(9, 3 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = axes.reshape(1, -1)

for row, case_id in enumerate(sample_ids):
    img = np.array(Image.open(os.path.join(images_dir, f"{case_id}_0000.png")))
    gt = np.array(Image.open(os.path.join(gt_dir, f"{case_id}.png")))
    pred = np.array(Image.open(os.path.join(pred_dir, f"{case_id}.png")))

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Original\n{case_id}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Ground Truth")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("Predicted")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/visualization_batch.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
!pip install nnunetv2 -q
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import os

nnUNet_raw = "/kaggle/working/nnUNet_raw"
nnUNet_preprocessed = "/kaggle/working/nnUNet_preprocessed"
nnUNet_results = "/kaggle/working/nnUNet_results"

for p in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(p, exist_ok=True)

os.environ["nnUNet_raw"] = nnUNet_raw
os.environ["nnUNet_preprocessed"] = nnUNet_preprocessed
os.environ["nnUNet_results"] = nnUNet_results

print(os.environ["nnUNet_raw"])

In [ ]:
model_input_dir = "/kaggle/input/models/saharfeki/nnunet/pytorch/default/1"
for root, dirs, files in os.walk(model_input_dir):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import os, shutil
import numpy as np
from PIL import Image
from glob import glob

dataset_dir = "/kaggle/working/nnUNet_raw/Dataset001_DFU"
for sub in ["imagesTr", "labelsTr"]:
    os.makedirs(os.path.join(dataset_dir, sub), exist_ok=True)

src_images = sorted(glob("/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_images/*.jpg"))
src_masks  = sorted(glob("/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_masks/*.png"))

img_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in src_images}
mask_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in src_masks}
common_ids = sorted(set(img_dict) & set(mask_dict))
print(f"{len(common_ids)} matched pairs found")

for case_id in common_ids:
    img = Image.open(img_dict[case_id]).convert("RGB")
    img.save(os.path.join(dataset_dir, "imagesTr", f"{case_id}_0000.png"))

    mask = Image.open(mask_dict[case_id]).convert("L")
    binary_arr = (np.array(mask) > 127).astype(np.uint8)
    Image.fromarray(binary_arr, mode="L").save(os.path.join(dataset_dir, "labelsTr", f"{case_id}.png"))

print("Conversion done:", len(common_ids), "cases")

In [ ]:
import json

dataset_json = {
    "channel_names": {"0": "R", "1": "G", "2": "B"},
    "labels": {
        "background": 0,
        "wound": 1
    },
    "numTraining": len(common_ids),
    "file_ending": ".png"
}

with open(os.path.join(dataset_dir, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=4)

print("dataset.json written")

In [ ]:
!nnUNetv2_extract_fingerprint -d 001
!nnUNetv2_plan_experiment -d 001

In [ ]:
trainer_name = "nnUNetTrainer_250epochs"  # match whatever trainer produced your checkpoint
config_folder = os.path.join(
    "/kaggle/working/nnUNet_results/Dataset001_DFU",
    f"{trainer_name}__nnUNetPlans__2d"
)
fold0_dir = os.path.join(config_folder, "fold_0")
os.makedirs(fold0_dir, exist_ok=True)

# copy plans + dataset json into the results config folder (required by nnUNetv2_predict)
shutil.copy(
    "/kaggle/working/nnUNet_preprocessed/Dataset001_DFU/nnUNetPlans.json",
    os.path.join(config_folder, "plans.json")
)
shutil.copy(
    "/kaggle/working/nnUNet_preprocessed/Dataset001_DFU/dataset_fingerprint.json",
    os.path.join(config_folder, "dataset_fingerprint.json")
)
shutil.copy(
    os.path.join(dataset_dir, "dataset.json"),
    os.path.join(config_folder, "dataset.json")
)

# copy your downloaded checkpoint
model_input_dir = "/kaggle/input/models/saharfeki/nnunet/pytorch/default/1"
checkpoint_file = [f for f in os.listdir(model_input_dir) if f.endswith(".pth")][0]
shutil.copy(
    os.path.join(model_input_dir, checkpoint_file),
    os.path.join(fold0_dir, "checkpoint_best.pth")
)

print("Config folder ready:", config_folder)
print(os.listdir(config_folder))
print(os.listdir(fold0_dir))

In [ ]:
import os
from PIL import Image

new_data_root = "/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes/train"
predict_input_dir = "/kaggle/working/new_data_imagesTs"
os.makedirs(predict_input_dir, exist_ok=True)

case_id_map = {}  # keeps track of original filename per case_id, useful for visualization later

for grade in ["Grade 1", "Grade 2", "Grade 3", "Grade 4"]:
    grade_dir = os.path.join(new_data_root, grade)
    for fname in os.listdir(grade_dir):
        if not fname.lower().endswith(".jpg"):
            continue
        case_id = f"{grade.replace(' ', '')}_{os.path.splitext(fname)[0]}"
        img = Image.open(os.path.join(grade_dir, fname)).convert("RGB")
        img.save(os.path.join(predict_input_dir, f"{case_id}_0000.png"))
        case_id_map[case_id] = (grade, fname)

print(f"Converted {len(case_id_map)} images into {predict_input_dir}")

In [ ]:
!nnUNetv2_predict -i /kaggle/working/new_data_imagesTs \
                   -o /kaggle/working/new_data_predictions \
                   -d 001 -c 2d -f 0 \
                   -tr nnUNetTrainer_250epochs \
                   -chk checkpoint_best.pth

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os, random

images_dir = "/kaggle/working/new_data_imagesTs"
pred_dir = "/kaggle/working/new_data_predictions"

case_ids = [f.replace("_0000.png", "") for f in os.listdir(images_dir) if f.endswith("_0000.png")]
pred_ids = set(f.replace(".png", "") for f in os.listdir(pred_dir) if f.endswith(".png"))

valid_ids = [c for c in case_ids if c in pred_ids]
print(f"{len(valid_ids)} cases available for visualization")

n_samples = 4
sample_ids = random.sample(valid_ids, min(n_samples, len(valid_ids)))

fig, axes = plt.subplots(len(sample_ids), 2, figsize=(7, 3.5 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = axes.reshape(1, -1)

for row, case_id in enumerate(sample_ids):
    img = np.array(Image.open(os.path.join(images_dir, f"{case_id}_0000.png")))
    pred = np.array(Image.open(os.path.join(pred_dir, f"{case_id}.png")))

    grade, orig_fname = case_id_map.get(case_id, ("?", case_id))

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Original ({grade})\n{orig_fname}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Predicted Mask")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/new_data_visualization.png", dpi=150, bbox_inches="tight")
plt.close()

from IPython.display import Image as IPImage, display
display(IPImage("/kaggle/working/new_data_visualization.png"))

# yolo

In [ ]:
!pip install -q ultralytics 
   
import os, glob, random, shutil  
import numpy as np 
import cv2
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [ ]:
IMAGES_DIR = '/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_images'
MASKS_DIR  = '/kaggle/input/datasets/pabodhamallawa/dfuc2022-train-release/DFUC2022_train_release/DFUC2022_train_masks'

BASE_DIR = '/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes'
GRADE_DIRS = {
    'train': {
        1: f'{BASE_DIR}/train/Grade 1', 2: f'{BASE_DIR}/train/Grade 2',
        3: f'{BASE_DIR}/train/Grade 3', 4: f'{BASE_DIR}/train/Grade 4',
    },
    'val': {
        1: f'{BASE_DIR}/val/Grade 1', 2: f'{BASE_DIR}/val/Grade 2',
        3: f'{BASE_DIR}/val/Grade 3', 4: f'{BASE_DIR}/val/Grade 4',
    },
    'test': {
        1: f'{BASE_DIR}/test/Grade 1', 2: f'{BASE_DIR}/test/Grade 2',
        3: f'{BASE_DIR}/test/Grade 3', 4: f'{BASE_DIR}/test/Grade 4',
    },
}

YOLO_ROOT = '/kaggle/working/yolo_dataset'
CROP_OUTPUT_DIR = '/kaggle/working/crops_yolo'
PAD_RATIO = 0.15
VAL_SPLIT = 0.2

os.makedirs(YOLO_ROOT, exist_ok=True)
os.makedirs(CROP_OUTPUT_DIR, exist_ok=True)

In [ ]:
image_files = sorted(glob.glob(f'{IMAGES_DIR}/*.jpg') + glob.glob(f'{IMAGES_DIR}/*.png'))
mask_files  = sorted(glob.glob(f'{MASKS_DIR}/*.jpg') + glob.glob(f'{MASKS_DIR}/*.png'))

print(f'{len(image_files)} images, {len(mask_files)} masks')

def get_mask_path(img_path):
    """Adjust this if mask naming convention differs (e.g. suffix _mask)."""
    stem = Path(img_path).stem
    for ext in ['.png', '.jpg']:
        candidate = f'{MASKS_DIR}/{stem}{ext}'
        if os.path.exists(candidate):
            return candidate
    return None

# sanity check pairing
missing = [f for f in image_files if get_mask_path(f) is None]
print(f'{len(missing)} images missing a matching mask')

In [ ]:
def mask_to_yolo_bbox(mask_path, img_w, img_h):
    """Returns normalized (x_center, y_center, w, h) or None if mask is empty."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask_bin = (mask > 127).astype(np.uint8)
    ys, xs = np.where(mask_bin > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    x_center = (x1 + x2) / 2 / img_w
    y_center = (y1 + y2) / 2 / img_h
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    return x_center, y_center, w, h

# quick visual check on a few samples
fig, axes = plt.subplots(1, 3, figsize=(15,5))
for i, img_path in enumerate(random.sample(image_files, 3)):
    mask_path = get_mask_path(img_path)
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    bbox = mask_to_yolo_bbox(mask_path, w, h)
    if bbox:
        xc, yc, bw, bh = bbox
        x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
        x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
        cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 3)
    axes[i].imshow(img); axes[i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
random.shuffle(image_files)
n_val = int(len(image_files) * VAL_SPLIT)
val_imgs = image_files[:n_val]
train_imgs = image_files[n_val:]

for split, files in [('train', train_imgs), ('val', val_imgs)]:
    img_out = f'{YOLO_ROOT}/images/{split}'
    lbl_out = f'{YOLO_ROOT}/labels/{split}'
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    skipped = 0
    for img_path in files:
        mask_path = get_mask_path(img_path)
        if mask_path is None:
            skipped += 1
            continue
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        bbox = mask_to_yolo_bbox(mask_path, w, h)
        if bbox is None:
            skipped += 1
            continue

        stem = Path(img_path).stem
        shutil.copy(img_path, f'{img_out}/{stem}.jpg')
        xc, yc, bw, bh = bbox
        with open(f'{lbl_out}/{stem}.txt', 'w') as f:
            f.write(f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n')

    print(f'{split}: {len(files)-skipped} written, {skipped} skipped (empty mask/missing pair)')

In [ ]:
yaml_content = f"""
path: {YOLO_ROOT}
train: images/train
val: images/val

names:
  0: wound
"""

with open(f'{YOLO_ROOT}/data.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

In [ ]:
model = YOLO('yolov8n.pt')  # swap to 'yolov5su.pt' if you specifically want v5

results = model.train(
    data=f'{YOLO_ROOT}/data.yaml',
    epochs=80,
    imgsz=640,
    optimizer='SGD',
    lr0=0.01,
    batch=16,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # HSV jitter
    translate=0.1, scale=0.5,             # translation/scaling
    fliplr=0.5, flipud=0.1,               # flips
    mosaic=1.0,                           # mosaic aug
    project='/kaggle/working/yolo_runs',
    name='wound_detector',
    seed=SEED,
    patience=15,
)

In [ ]:
 metrics = model.val(data=f'{YOLO_ROOT}/data.yaml')
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.mp:.3f}, Recall: {metrics.box.mr:.3f}")

In [ ]:
best_weights = f'/kaggle/working/yolo_runs/wound_detector/weights/best.pt'
yolo_model = YOLO(best_weights)

In [ ]:
def get_wound_box(img_path, conf_threshold=0.25, iou_merge_threshold=0.3):
    """
    Runs YOLO inference, returns a single (x1,y1,x2,y2) box in pixel coords.
    If multiple detections: merges boxes that overlap significantly (likely same wound,
    split detection like the paper's patient-4 case), otherwise picks highest confidence.
    Returns None if no detection (caller should fall back to full image).
    """
    result = yolo_model.predict(img_path, conf=conf_threshold, verbose=False)[0]
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        return None

    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()

    if len(xyxy) == 1:
        return tuple(xyxy[0])

    # multiple boxes: check overlap via IoU
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2-x1) * max(0, y2-y1)
        area1 = (b1[2]-b1[0])*(b1[3]-b1[1])
        area2 = (b2[2]-b2[0])*(b2[3]-b2[1])
        return inter / (area1 + area2 - inter + 1e-6)

    # sort by confidence, check if top-2 overlap enough to merge
    order = np.argsort(-confs)
    top_box = xyxy[order[0]]
    if len(order) > 1:
        second_box = xyxy[order[1]]
        if iou(top_box, second_box) > iou_merge_threshold:
            # merge: union of both boxes
            merged = (
                min(top_box[0], second_box[0]), min(top_box[1], second_box[1]),
                max(top_box[2], second_box[2]), max(top_box[3], second_box[3])
            )
            return merged

    # no significant overlap: just take highest-confidence box
    return tuple(top_box)

In [ ]:
def crop_with_padding(img_path, pad_ratio=PAD_RATIO):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    box = get_wound_box(img_path)
    if box is None:
        return img, True  # fallback: full image

    x1, y1, x2, y2 = box
    bw, bh = x2-x1, y2-y1
    pad_x, pad_y = bw*pad_ratio, bh*pad_ratio
    x1 = max(0, int(x1-pad_x)); y1 = max(0, int(y1-pad_y))
    x2 = min(w, int(x2+pad_x)); y2 = min(h, int(y2+pad_y))

    crop = img[y1:y2, x1:x2]
    return crop, False

In [ ]:
sample_paths = []

for grade, folder in GRADE_DIRS['train'].items():
    files = glob.glob(f"{folder}/*.jpg") + glob.glob(f"{folder}/*.png")
    selected = random.sample(files, min(2, len(files)))

    for f in selected:
        sample_paths.append((f, grade))

fig, axes = plt.subplots(len(sample_paths), 3,
                         figsize=(15, 4*len(sample_paths)))

if len(sample_paths) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, (p, grade) in enumerate(sample_paths):

    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)

    crop, fell_back = crop_with_padding(p)

    # Copy for drawing the YOLO prediction
    img_box = img.copy()

    box = get_wound_box(p)

    if box is not None:
        x1, y1, x2, y2 = map(int, box)

        # Draw red rectangle
        cv2.rectangle(
            img_box,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),   # RGB Red
            3
        )

    # -------------------------
    # Column 1 : Original image
    # -------------------------
    axes[i,0].imshow(img)
    axes[i,0].set_title(f"Original\n{grade}")
    axes[i,0].axis("off")

    # -------------------------
    # Column 2 : YOLO detection
    # -------------------------
    axes[i,1].imshow(img_box)

    if box is None:
        axes[i,1].set_title("YOLO Detection\n(No detection)",
                            color="red")
    else:
        axes[i,1].set_title("YOLO Bounding Box")

    axes[i,1].axis("off")

    # -------------------------
    # Column 3 : Crop
    # -------------------------
    axes[i,2].imshow(crop)

    if fell_back:
        axes[i,2].set_title("Crop\n(Fallback = Full Image)",
                            color="red")
    else:
        axes[i,2].set_title("Cropped ROI")

    axes[i,2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sample_dir = "/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes/train/Grade 3"

all_images = (
    glob.glob(f"{sample_dir}/*.jpg") +
    glob.glob(f"{sample_dir}/*.png") +
    glob.glob(f"{sample_dir}/*.jpeg")
)

sample_paths = random.sample(all_images, min(6, len(all_images)))

# 3 columns: Original | YOLO Detection | Crop
fig, axes = plt.subplots(len(sample_paths), 3, figsize=(15, 4 * len(sample_paths)))

# Handle the case of a single image
if len(sample_paths) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, p in enumerate(sample_paths):

    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)

    crop, fell_back = crop_with_padding(p)

    # Copy image for drawing YOLO box
    img_box = img.copy()

    box = get_wound_box(p)

    if box is not None:
        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(
            img_box,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),   # RGB Red
            3
        )

    filename = os.path.basename(p)

    # -------------------------
    # Column 1 : Original image
    # -------------------------
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Original\n{filename}")
    axes[i, 0].axis("off")

    # -------------------------
    # Column 2 : YOLO detection
    # -------------------------
    axes[i, 1].imshow(img_box)

    if box is None:
        axes[i, 1].set_title("YOLO Detection\n(No detection)", color="red")
    else:
        axes[i, 1].set_title("YOLO Bounding Box")

    axes[i, 1].axis("off")

    # -------------------------
    # Column 3 : Crop
    # -------------------------
    axes[i, 2].imshow(crop)

    if fell_back:
        axes[i, 2].set_title("Crop\n(Fallback = Full Image)", color="red")
    else:
        axes[i, 2].set_title("Cropped ROI")

    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

# 4 models comparision

In [ ]:
!pip install -q --no-deps segmentation_models_pytorch ttach
!pip install -q albumentations
!pip install -q ultralytics

import segmentation_models_pytorch as smp
from ultralytics import YOLO
import torch, cv2, numpy as np
import matplotlib.pyplot as plt
import glob, random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Adjust paths to match your actual Kaggle input dataset names
AUG_MODEL_PATH = '/kaggle/input/models/saharfeki/unet-aug-0-82/pytorch/default/1/0.82 model with augmentation.pth'
BG_MODEL_PATH  = '/kaggle/input/models/saharfeki/unet-bg-removal/pytorch/default/1/best_model bg_remove.pth'
YOLO_PATH       = '/kaggle/input/models/saharfeki/yolov8/pytorch/default/1/yolov8n.pt'

IMG_SIZE = 512  # EDIT to match your training resolution

def build_model():
    model = smp.Unet(

        encoder_name="resnet34",
        encoder_weights=None,   # weights come from checkpoint, not ImageNet, at load time
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model

def load_checkpoint(model, path, device):
    ckpt = torch.load(path, map_location=device)
    # handle both raw state_dict and dict-wrapped checkpoints
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt
    # strip "module." prefix if saved from DataParallel
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()
    return model

model_bg  = load_checkpoint(build_model(), BG_MODEL_PATH, device)
model_aug = load_checkpoint(build_model(), AUG_MODEL_PATH, device)
print("Both models loaded.")

# --- Model 3: YOLO detector ---
yolo_model = YOLO(YOLO_PATH)

print("yolo model loaded.")

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_and_preprocess(img_path, size=256):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig = img.copy()
    img_resized = cv2.resize(img, (size, size))
    img_norm = img_resized.astype(np.float32) / 255.0
    img_norm = (img_norm - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0).float()
    return tensor, orig

def predict_mask(model, img_path, threshold=0.5, size=256):
    tensor, orig = load_and_preprocess(img_path, size)
    model.eval()
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor.to(device)))
    mask = pred.squeeze().cpu().numpy()
    mask_bin = (mask > threshold).astype(np.uint8)
    mask_bin = cv2.resize(mask_bin, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask_bin, orig

def predict_yolo_box(img_path, conf_threshold=0.25):
    result = yolo_model.predict(img_path, conf=conf_threshold, verbose=False)[0]
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return None
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()
    best_idx = np.argmax(confs)
    return tuple(xyxy[best_idx]), confs[best_idx]

In [ ]:
import os
import random
import glob
import time
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss

In [ ]:
# ── Cell 13: Load 4-channel U-Net checkpoint ────────────────────────────────

def build_unet_4ch(encoder="resnet34", encoder_weights=None, classes=1):
    model = smp.Unet(
        encoder_name=encoder, encoder_weights=encoder_weights,
        in_channels=3, classes=classes, activation=None,
    )
    old_conv = model.encoder.conv1
    new_conv = nn.Conv2d(
        in_channels=4, out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, bias=(old_conv.bias is not None)
    )
    with torch.no_grad():
        new_conv.weight[:, :3] = old_conv.weight
        new_conv.weight[:, 3:4] = old_conv.weight.mean(dim=1, keepdim=True)
        if old_conv.bias is not None:
            new_conv.bias[:] = old_conv.bias
    model.encoder.conv1 = new_conv
    return model

NEW_MODEL_PATH = "/kaggle/input/models/saharfeki/unet-improved/pytorch/default/1/unet improved.pth"  # EDIT path

model_new = build_unet_4ch()
model_new = load_checkpoint(model_new, NEW_MODEL_PATH, device)  # reuses your Cell 4 loader
print("4-channel model loaded.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def overlay_mask(image, mask, color=(255, 0, 0), alpha=0.4):
    colored = np.zeros_like(image)
    colored[mask.astype(bool)] = color
    return cv2.addWeighted(colored, alpha, image, 1 - alpha, 0)

def predict_mask_4ch(model_new, model_bg, img_path, size=256, threshold=0.5):
    """4th channel = bg-removal model's predicted mask. Adjust if that's not your setup."""
    bg_mask, orig = predict_mask(model_bg, img_path, threshold=threshold, size=size)

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (size, size))
    img_norm = img_resized.astype(np.float32) / 255.0
    img_norm = (img_norm - IMAGENET_MEAN) / IMAGENET_STD

    mask_ch = cv2.resize(bg_mask, (size, size), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    four_ch = np.dstack([img_norm, mask_ch])
    tensor = torch.from_numpy(four_ch.transpose(2, 0, 1)).unsqueeze(0).float()

    model_new.eval()
    with torch.no_grad():
        pred = torch.sigmoid(model_new(tensor.to(device)))
    mask = pred.squeeze().cpu().numpy()
    mask_bin = (mask > threshold).astype(np.uint8)
    mask_bin = cv2.resize(mask_bin, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask_bin, orig

def visualize_all(img_path, size=256, conf_threshold=0.25):
    mask_bg, orig   = predict_mask(model_bg, img_path, size=size)
    mask_aug, _     = predict_mask(model_aug, img_path, size=size)
    mask_new, _     = predict_mask_4ch(model_new, model_bg, img_path, size=size)
    yolo_res        = predict_yolo_box(img_path, conf_threshold=conf_threshold)

    overlay_bg  = overlay_mask(orig, mask_bg,  color=(255, 0, 0))   # red
    overlay_aug = overlay_mask(orig, mask_aug, color=(0, 255, 0))   # green
    overlay_new = overlay_mask(orig, mask_new, color=(0, 0, 255))   # blue

    yolo_img = orig.copy()
    if yolo_res is not None:
        (x1, y1, x2, y2), conf = yolo_res
        cv2.rectangle(yolo_img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 255, 0), 3)
        cv2.putText(yolo_img, f"{conf:.2f}", (int(x1), max(int(y1) - 10, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)

    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    axes[0,0].imshow(orig);          axes[0,0].set_title("Original");             axes[0,0].axis("off")
    axes[0,1].imshow(mask_bg,  cmap="gray"); axes[0,1].set_title("BG-removal mask");      axes[0,1].axis("off")
    axes[0,2].imshow(mask_aug, cmap="gray"); axes[0,2].set_title("Aug mask");             axes[0,2].axis("off")
    axes[0,3].imshow(mask_new, cmap="gray"); axes[0,3].set_title("4-channel model mask"); axes[0,3].axis("off")

    axes[1,0].imshow(yolo_img);      axes[1,0].set_title("YOLO detection");        axes[1,0].axis("off")
    axes[1,1].imshow(overlay_bg);    axes[1,1].set_title("BG-removal overlay");    axes[1,1].axis("off")
    axes[1,2].imshow(overlay_aug);   axes[1,2].set_title("Aug overlay");           axes[1,2].axis("off")
    axes[1,3].imshow(overlay_new);   axes[1,3].set_title("4-channel overlay");     axes[1,3].axis("off")

    plt.tight_layout()
    plt.show()

# ---------- run on a few random samples ----------
IMG_DIR = "/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes/train/Grade 4"  # EDIT to your actual images folder
sample_paths = random.sample(glob.glob(f"{IMG_DIR}/*.*"), k=min(3, len(glob.glob(f"{IMG_DIR}/*.*"))))

for p in sample_paths:
    print(p)
    visualize_all(p, size=IMG_SIZE)

# classification

In [ ]:
!pip install -q --no-deps segmentation_models_pytorch ttach
!pip install -q albumentations
!pip install -q ultralytics
!pip install -q timm

import os, glob, random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import segmentation_models_pytorch as smp
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

In [ ]:
BASE_DIR = '/kaggle/input/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes'
GRADE_DIRS = {
    'train': {1:f'{BASE_DIR}/train/Grade 1', 2:f'{BASE_DIR}/train/Grade 2',
               3:f'{BASE_DIR}/train/Grade 3', 4:f'{BASE_DIR}/train/Grade 4'},
    'val':   {1:f'{BASE_DIR}/valid/Grade 1',   2:f'{BASE_DIR}/valid/Grade 2',
               3:f'{BASE_DIR}/valid/Grade 3',   4:f'{BASE_DIR}/valid/Grade 4'},
    'test':  {1:f'{BASE_DIR}/test/Grade 1',  2:f'{BASE_DIR}/test/Grade 2',
               3:f'{BASE_DIR}/test/Grade 3',  4:f'{BASE_DIR}/test/Grade 4'},
}

IMG_SIZE = 224
NUM_CLASSES = 4
PAD_RATIO = 0.15
BATCH_SIZE = 32

In [ ]:
def load_checkpoint(model, path, device):
    ckpt = torch.load(path, map_location=device)
    # handle both raw state_dict and dict-wrapped checkpoints
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt
    # strip "module." prefix if saved from DataParallel
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()
    return model

In [ ]:
def build_unet_4ch(encoder="resnet34", encoder_weights=None, classes=1):
    model = smp.Unet(
        encoder_name=encoder, encoder_weights=encoder_weights,
        in_channels=3, classes=classes, activation=None,
    )
    old_conv = model.encoder.conv1
    new_conv = nn.Conv2d(
        in_channels=4, out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, bias=(old_conv.bias is not None)
    )
    with torch.no_grad():
        new_conv.weight[:, :3] = old_conv.weight
        new_conv.weight[:, 3:4] = old_conv.weight.mean(dim=1, keepdim=True)
        if old_conv.bias is not None:
            new_conv.bias[:] = old_conv.bias
    model.encoder.conv1 = new_conv
    return model

NEW_MODEL_PATH = "/kaggle/input/models/saharfeki/unet-improved/pytorch/default/1/unet improved.pth"  # EDIT path

model_new = build_unet_4ch()
unet_model = load_checkpoint(model_new, NEW_MODEL_PATH, DEVICE)  # reuses your Cell 4 loader
print("4-channel model loaded.")

In [ ]:
BG_MODEL_PATH  = '/kaggle/input/models/saharfeki/unet-bg-removal/pytorch/default/1/best_model bg_remove.pth'
def build_model():
    model = smp.Unet(

        encoder_name="resnet34",
        encoder_weights=None,   # weights come from checkpoint, not ImageNet, at load time
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model
model_bg  = load_checkpoint(build_model(), BG_MODEL_PATH, DEVICE)

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict_mask_and_crop(img_path, unet_size=256, pad_ratio=PAD_RATIO, threshold=0.5, min_bbox_size=5):
    """
    Returns (cropped_rgb, cropped_mask, fell_back).
    If the U-Net fails to detect a wound (empty mask OR a degenerate/near-zero-area mask),
    fall back to using the full original image as the "crop" — many dataset images are
    already pre-cropped to the wound, so this is a safe default.
    """
    img = cv2.imread(img_path) 
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    bg_mask, _ = predict_mask(model_bg, img_path, threshold=threshold, size=unet_size)
    mask_ch = cv2.resize(bg_mask, (unet_size, unet_size),
                          interpolation=cv2.INTER_NEAREST).astype(np.float32)

    img_resized = cv2.resize(img, (unet_size, unet_size)).astype(np.float32) / 255.0
    img_norm = (img_resized - IMAGENET_MEAN) / IMAGENET_STD

    four_ch = np.dstack([img_norm, mask_ch])
    tensor = torch.from_numpy(four_ch.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        pred = torch.sigmoid(unet_model(tensor))
    mask = pred.squeeze().cpu().numpy()
    mask_bin = (mask > threshold).astype(np.uint8)
    mask_full = cv2.resize(mask_bin, (w, h), interpolation=cv2.INTER_NEAREST)

    ys, xs = np.where(mask_full > 0)

    # Case 1: completely empty mask -> fallback
    if len(xs) == 0 or len(ys) == 0:
        return img, np.zeros((h, w), dtype=np.uint8), True

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    bw, bh = x2 - x1, y2 - y1

    # Case 2: degenerate/near-zero-area mask (model "detected" noise, not a real region) -> fallback
    if bw < min_bbox_size or bh < min_bbox_size:
        return img, np.zeros((h, w), dtype=np.uint8), True

    # Normal case: valid bounding box -> pad and crop
    pad_x, pad_y = int(bw * pad_ratio), int(bh * pad_ratio)
    x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x); y2 = min(h, y2 + pad_y)

    crop_img = img[y1:y2, x1:x2]
    crop_mask = mask_full[y1:y2, x1:x2]
    return crop_img, crop_mask, False

In [ ]:
def min_max_normalize(image_rgb):
    img = image_rgb.astype(np.float32)
    img_min, img_max = img.min(), img.max()
    if img_max - img_min < 1e-6:
        return image_rgb.astype(np.uint8)
    norm = (img - img_min) / (img_max - img_min)
    return (norm * 255).astype(np.uint8)


def fill_holes_flood(mask):
    """Border flood-fill: fills only holes fully unreachable from the image edge."""
    h, w = mask.shape
    flood_fill_mask = np.zeros((h + 2, w + 2), np.uint8)
    mask_copy = mask.copy()
    cv2.floodFill(mask_copy, flood_fill_mask, (0, 0), 255)
    background_reachable_inv = cv2.bitwise_not(mask_copy)
    filled = cv2.bitwise_or(mask, background_reachable_inv)
    return filled


def fill_via_convex_hull(mask):
    """
    Fills concave bays/notches that touch the border or connect to true
    background through a thin channel (which fill_holes_flood cannot fix,
    since it only fills fully-enclosed holes).
    Replaces each connected component with its convex hull — guarantees no
    notch survives regardless of how it connects to the outside background.
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull_mask = np.zeros_like(mask)
    for cnt in contours:
        if cv2.contourArea(cnt) < 50:   # skip tiny noise contours
            continue
        hull = cv2.convexHull(cnt)
        cv2.drawContours(hull_mask, [hull], -1, 255, thickness=cv2.FILLED)
    return hull_mask


def keep_components_above_area(mask, min_area_frac=0.02):
    h, w = mask.shape
    min_area = min_area_frac * h * w
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == label] = 255
    return cleaned


def get_skin_mask(image_rgb, median_kernel=15, min_area_frac=0.02, closing_frac=0.05):
    """
    Background removal via color-space thresholding.
    - Min-max normalize image intensities first
    - Otsu threshold on Cr (YCbCr) and a* (CIELAB), combined with OR
    - Median filter + morphological closing (kernel scaled to image size)
    - Border flood-fill for fully-enclosed gaps
    - Convex-hull fill per component to eliminate any remaining bay/notch,
      including ones connected to background through the image border
      (fixes wound-bed cavities being carved out as "background")
    - Area-threshold component filtering to keep all significant blobs
    """
    h, w = image_rgb.shape[:2]
    norm_img = min_max_normalize(image_rgb)

    ycrcb = cv2.cvtColor(norm_img, cv2.COLOR_RGB2YCrCb)
    cr = ycrcb[:, :, 1]
    _, cr_mask = cv2.threshold(cr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    lab = cv2.cvtColor(norm_img, cv2.COLOR_RGB2LAB)
    a_ch = lab[:, :, 1]
    _, a_mask = cv2.threshold(a_ch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    combined = cv2.bitwise_or(cr_mask, a_mask)

    k = int(min(h, w) * 0.02) | 1   # odd kernel, scaled to image size
    k = max(k, median_kernel)
    cleaned = cv2.medianBlur(combined, k)

    # Closing kernel scaled to image size — big enough to bridge large notches
    closing_kernel = max(21, int(min(h, w) * closing_frac)) | 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (closing_kernel, closing_kernel))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)

    cleaned = fill_holes_flood(cleaned)
    cleaned = keep_components_above_area(cleaned, min_area_frac)
    cleaned = fill_via_convex_hull(cleaned)          # NEW — guarantees no notch survives

    return cleaned


def apply_background_removal(image_rgb, median_kernel=15, min_area_frac=0.02, closing_frac=0.05):
    mask = get_skin_mask(image_rgb, median_kernel=median_kernel,
                          min_area_frac=min_area_frac, closing_frac=closing_frac)
    mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB) // 255
    masked = (image_rgb * mask_3ch).astype(np.uint8)
    return masked, mask



In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_and_preprocess(img_path, size=256):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig = img.copy()
    img_resized = cv2.resize(img, (size, size))
    img_norm = img_resized.astype(np.float32) / 255.0
    img_norm = (img_norm - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0).float()
    return tensor, orig

def predict_mask(model, img_path, threshold=0.5, size=256):
    tensor, orig = load_and_preprocess(img_path, size)
    model.eval()
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor.to(DEVICE)))
    mask = pred.squeeze().cpu().numpy()
    mask_bin = (mask > threshold).astype(np.uint8)
    mask_bin = cv2.resize(mask_bin, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask_bin, orig

In [ ]:
import random
import matplotlib.pyplot as plt

def visualize_random_batch(n_samples=4, seed=None):
    if seed is not None:
        random.seed(seed)

    # Gather all (path, split, grade) tuples
    all_files = []
    for split, grade_dict in GRADE_DIRS.items():
        for grade, dir_path in grade_dict.items():
            files = glob.glob(f'{dir_path}/*.jpg') + glob.glob(f'{dir_path}/*.png')
            all_files.extend([(fp, split, grade) for fp in files])


    samples = random.sample(all_files, min(n_samples, len(all_files)))

    fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4 * len(samples)))
    if len(samples) == 1:
        axes = axes[None, :]  # keep 2D indexing consistent

    for row, (fp, split, grade) in enumerate(samples):
        img = cv2.imread(fp)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        foot_mask = get_skin_mask(img)
        crop_img, crop_mask, fell_back = predict_mask_and_crop(fp)

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'{split} / Grade{grade}\nOriginal')

        axes[row, 1].imshow(foot_mask, cmap='gray')
        axes[row, 1].set_title('Foot/skin mask\n(4th channel input)')

        axes[row, 2].imshow(img)
        axes[row, 2].imshow(crop_mask if crop_mask.shape == img.shape[:2] else
                             cv2.resize(crop_mask, (img.shape[1], img.shape[0]),
                                        interpolation=cv2.INTER_NEAREST),
                             cmap='jet', alpha=0.4) if False else None
        # show predicted wound mask on the crop itself instead (more meaningful)
        axes[row, 2].imshow(crop_img)
        axes[row, 2].imshow(crop_mask, cmap='jet', alpha=0.4)
        axes[row, 2].set_title(f'Crop + predicted wound mask\nfallback={fell_back}')

        axes[row, 3].imshow(crop_img)
        axes[row, 3].set_title(f'Final crop\n{crop_img.shape[1]}x{crop_img.shape[0]}')

        for ax in axes[row]:
            ax.axis('off')

    plt.tight_layout()
    plt.show()

visualize_random_batch(n_samples=4, seed=None)

In [ ]:
from tqdm.auto import tqdm
import time

CROP_DIR = '/kaggle/working/crops_4ch'
os.makedirs(CROP_DIR, exist_ok=True)

def generate_4ch_crops(split_name):
    fallback_count, total_count, error_count = 0, 0, 0
    manifest = []
    errors = []

    # Build full file list first so tqdm knows the total and we can show grade context
    all_files = []
    for grade, dir_path in GRADE_DIRS[split_name].items():
        files = glob.glob(f'{dir_path}/*.jpg') + glob.glob(f'{dir_path}/*.png')
        all_files.extend([(fp, grade) for fp in files])

    print(f"\n=== {split_name}: {len(all_files)} files found across {len(GRADE_DIRS[split_name])} grades ===")

    pbar = tqdm(all_files, desc=f'{split_name}', unit='img')
    start_time = time.time()

    for fp, grade in pbar:
        out_dir = f'{CROP_DIR}/{split_name}/Grade{grade}'
        os.makedirs(out_dir, exist_ok=True)

        try:
            crop_img, crop_mask, fell_back = predict_mask_and_crop(fp)

            if crop_img is None or crop_img.size == 0 or 0 in crop_img.shape[:2]:
                raise ValueError(f"Empty crop_img with shape {getattr(crop_img, 'shape', None)}")
            if crop_mask is None or crop_mask.size == 0 or 0 in crop_mask.shape[:2]:
                raise ValueError(f"Empty crop_mask with shape {getattr(crop_mask, 'shape', None)}")

            crop_img_r = cv2.resize(crop_img, (IMG_SIZE, IMG_SIZE))
            crop_mask_r = cv2.resize(crop_mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

            stacked = np.dstack([crop_img_r, crop_mask_r.astype(np.uint8) * 255])
            fname = os.path.splitext(os.path.basename(fp))[0] + '.npy'
            out_path = f'{out_dir}/{fname}'
            np.save(out_path, stacked)

            manifest.append({'path': out_path, 'grade': grade, 'split': split_name, 'fallback': fell_back})
            fallback_count += fell_back
            total_count += 1

        except Exception as e:
            error_count += 1
            errors.append((fp, str(e)))
            pbar.write(f"  [SKIPPED] {os.path.basename(fp)} -> {e}")
            continue

        # Live diagnostics in the progress bar itself
        elapsed = time.time() - start_time
        rate = total_count / elapsed if elapsed > 0 else 0
        pbar.set_postfix({
            'fallback%': f'{100*fallback_count/max(total_count,1):.1f}',
            'errors': error_count,
            'img/s': f'{rate:.1f}'
        })

    print(f'{split_name} done: {total_count} saved, {fallback_count} fallbacks '
          f'({100*fallback_count/max(total_count,1):.1f}%), {error_count} errors, '
          f'{time.time()-start_time:.1f}s total')

    if errors:
        print(f"  {len(errors)} files failed:")
        for fp, err in errors[:20]:  # cap printed list to avoid flooding output
            print(f"    {fp}: {err}")
        if len(errors) > 20:
            print(f"    ...and {len(errors)-20} more")

    return manifest, errors


full_manifest = []
all_errors = []
for split in ['train', 'val', 'test']:
    manifest, errors = generate_4ch_crops(split)
    full_manifest.extend(manifest)
    all_errors.extend(errors)

manifest_df = pd.DataFrame(full_manifest)
manifest_df.to_csv(f'{CROP_DIR}/manifest.csv', index=False)

print("\n=== Final summary ===")
print(manifest_df.groupby(['split', 'grade']).size().unstack())
print(f"\nOverall fallback rate: {100*manifest_df['fallback'].mean():.1f}%")
print(f"Total files skipped due to errors: {len(all_errors)}")

In [ ]:
from tqdm.auto import tqdm
import time

CROP_DIR = '/kaggle/working/crops_4ch'
os.makedirs(CROP_DIR, exist_ok=True)

def generate_4ch_crops(split_name):
    fallback_skipped_count, total_count, error_count = 0, 0, 0
    manifest = []
    errors = []
    fallback_by_grade = {g: 0 for g in GRADE_DIRS[split_name].keys()}

    # Build full file list first so tqdm knows the total and we can show grade context
    all_files = []
    for grade, dir_path in GRADE_DIRS[split_name].items():
        files = glob.glob(f'{dir_path}/*.jpg') + glob.glob(f'{dir_path}/*.png')
        all_files.extend([(fp, grade) for fp in files])

    print(f"\n=== {split_name}: {len(all_files)} files found across {len(GRADE_DIRS[split_name])} grades ===")
    pbar = tqdm(all_files, desc=f'{split_name}', unit='img')
    start_time = time.time()

    for fp, grade in pbar:
        out_dir = f'{CROP_DIR}/{split_name}/Grade{grade}'
        os.makedirs(out_dir, exist_ok=True)
        try:
            crop_img, crop_mask, fell_back = predict_mask_and_crop(fp)

            # Skip entirely if the bg-removal model fell back (no valid mask detected)
            if fell_back:
                fallback_skipped_count += 1
                fallback_by_grade[grade] += 1
                continue

            if crop_img is None or crop_img.size == 0 or 0 in crop_img.shape[:2]:
                raise ValueError(f"Empty crop_img with shape {getattr(crop_img, 'shape', None)}")
            if crop_mask is None or crop_mask.size == 0 or 0 in crop_mask.shape[:2]:
                raise ValueError(f"Empty crop_mask with shape {getattr(crop_mask, 'shape', None)}")

            crop_img_r = cv2.resize(crop_img, (IMG_SIZE, IMG_SIZE))
            crop_mask_r = cv2.resize(crop_mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            stacked = np.dstack([crop_img_r, crop_mask_r.astype(np.uint8) * 255])

            fname = os.path.splitext(os.path.basename(fp))[0] + '.npy'
            out_path = f'{out_dir}/{fname}'
            np.save(out_path, stacked)

            manifest.append({'path': out_path, 'grade': grade, 'split': split_name})
            total_count += 1

        except Exception as e:
            error_count += 1
            errors.append((fp, str(e)))
            pbar.write(f"  [SKIPPED-ERROR] {os.path.basename(fp)} -> {e}")
            continue

        # Live diagnostics in the progress bar itself
        elapsed = time.time() - start_time
        rate = total_count / elapsed if elapsed > 0 else 0
        pbar.set_postfix({
            'kept': total_count,
            'fallback_skip': fallback_skipped_count,
            'errors': error_count,
            'img/s': f'{rate:.1f}'
        })

    print(f'{split_name} done: {total_count} saved (mask found), '
          f'{fallback_skipped_count} skipped (fallback, no mask), '
          f'{error_count} errors, {time.time()-start_time:.1f}s total')

    print(f"  Fallback-skipped by grade: {fallback_by_grade}")

    if errors:
        print(f"  {len(errors)} files failed:")
        for fp, err in errors[:20]:
            print(f"    {fp}: {err}")
        if len(errors) > 20:
            print(f"    ...and {len(errors)-20} more")

    return manifest, errors, fallback_by_grade

full_manifest = []
all_errors = []
all_fallback_by_grade = {}

for split in ['train', 'val', 'test']:
    manifest, errors, fallback_by_grade = generate_4ch_crops(split)
    full_manifest.extend(manifest)
    all_errors.extend(errors)
    all_fallback_by_grade[split] = fallback_by_grade

manifest_df = pd.DataFrame(full_manifest)
manifest_df.to_csv(f'{CROP_DIR}/manifest.csv', index=False)

print("\n=== Final summary (kept, mask-only samples) ===")
print(manifest_df.groupby(['split', 'grade']).size().unstack())

print("\n=== Fallback-skipped counts by split & grade ===")
fallback_df = pd.DataFrame(all_fallback_by_grade).T  # rows=split, cols=grade
print(fallback_df)

print(f"\nTotal files skipped due to fallback: {sum(v for d in all_fallback_by_grade.values() for v in d.values())}")
print(f"Total files skipped due to errors: {len(all_errors)}")

**Prepare classifier**

In [ ]:
# ============================================================
# RGB-ONLY ABLATION (3-channel, no mask channel)
# Reuses the same crops (.npy files already have RGB in ch 0:3),
# just ignores channel 4 at load time.
# ============================================================

MEAN_3CH = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD_3CH  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def normalize_3ch(tensor):
    """tensor: (3,H,W) uint8 -> normalized float"""
    tensor = tensor.float() / 255.0
    return (tensor - MEAN_3CH) / STD_3CH

class WoundDatasetRGB(Dataset):
    """Same crops as WoundDataset4Ch, but drops the mask channel entirely."""
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        stacked = np.load(row['path'])  # (H,W,4) uint8
        label = row['grade'] - 1
        rgb = stacked[:, :, :3]  # ignore channel 4 entirely

        if self.transform:
            augmented = self.transform(image=rgb)
            rgb_t = augmented['image']  # (3,H,W) tensor from ToTensorV2
        else:
            rgb_t = torch.from_numpy(rgb.transpose(2, 0, 1))

        rgb_t = normalize_3ch(rgb_t)
        return rgb_t, label

# Standard transforms, no additional_targets needed since there's no mask to sync
train_transform_rgb = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    ToTensorV2()
])
val_transform_rgb = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    ToTensorV2()
])

# Use the SAME manifest/splits as your current 4-channel run for a fair comparison
train_df = manifest_df[manifest_df['split']=='train']
val_df   = manifest_df[manifest_df['split']=='val']
test_df  = manifest_df[manifest_df['split']=='test']
train_ds_rgb = WoundDatasetRGB(train_df, train_transform_rgb)
val_ds_rgb   = WoundDatasetRGB(val_df, val_transform_rgb)
test_ds_rgb  = WoundDatasetRGB(test_df, val_transform_rgb)

train_loader_rgb = DataLoader(train_ds_rgb, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_rgb   = DataLoader(val_ds_rgb, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_rgb  = DataLoader(test_ds_rgb, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def freeze_all_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

def unfreeze_fraction(model, fraction):
    params = list(model.backbone.parameters())
    n_unfreeze = int(len(params) * fraction)
    for p in params[:-n_unfreeze]:
        p.requires_grad = False
    for p in params[-n_unfreeze:]:
        p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True

def build_discriminative_optimizer(model, head_lr, backbone_lr):
    """Two param groups: head trains faster than the unfrozen backbone slice.
    Only includes params with requires_grad=True (so frozen layers are excluded)."""
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params = [p for p in model.head.parameters() if p.requires_grad]
    param_groups = []
    if backbone_params:
        param_groups.append({'params': backbone_params, 'lr': backbone_lr})
    if head_params:
        param_groups.append({'params': head_params, 'lr': head_lr})
    return optim.AdamW(param_groups, weight_decay=1e-4)

BACKBONE_MAP = {
    'resnet50v2': 'resnetv2_50x1_bit.goog_in21k',

}
class WoundClassifierRGB(nn.Module):
    """Same as WoundClassifier4Ch but WITHOUT the conv1 4-channel surgery —
    standard pretrained 3-channel backbone, untouched."""
    def __init__(self, backbone_key, num_classes=4, extra_dense=False, dense_dim=256):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE_MAP[backbone_key], pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        if extra_dense:
            self.head = nn.Sequential(
                nn.Linear(feat_dim, dense_dim), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(dense_dim, num_classes)
            )
        else:
            self.head = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)


def train_one_epoch_rgb(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, labels in loader:
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += x.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate_rgb(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for x, labels in loader:
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        outputs = model(x)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * x.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += x.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    return total_loss/total, correct/total, f1_macro, all_preds, all_labels


def run_three_phase_training_rgb(backbone_key, extra_dense, tag, patience=7, epochs_per_phase=20):
    print(f'\n=== Training {backbone_key} | extra_dense={extra_dense} ===')
    model = WoundClassifierRGB(backbone_key, NUM_CLASSES, extra_dense).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    history = []
    best_val_f1 = -1
    best_state = None
    best_epoch_info = None
    epochs_no_improve = 0

    def run_phase(phase_num, opt, scheduler, epochs_no_improve, best_val_f1, best_state, best_epoch_info):
        for epoch in range(epochs_per_phase):
            tr_loss, tr_acc = train_one_epoch_rgb(model, train_loader_rgb, opt, criterion)
            val_loss, val_acc, val_f1, _, _ = evaluate_rgb(model, val_loader_rgb, criterion)
            if scheduler is not None:
                scheduler.step()
            history.append({
                'phase': phase_num, 'epoch': epoch,
                'train_loss': tr_loss, 'train_acc': tr_acc,
                'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1,
                'lr_head': opt.param_groups[-1]['lr'],
                'lr_backbone': opt.param_groups[0]['lr'] if len(opt.param_groups) > 1 else None
            })
            print(f'P{phase_num} E{epoch}: train_acc={tr_acc:.3f} val_acc={val_acc:.3f} val_f1={val_f1:.3f} '
                  f'lr_head={opt.param_groups[-1]["lr"]:.2e}')
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_epoch_info = {'phase': phase_num, 'epoch': epoch, 'val_f1': val_f1, 'val_acc': val_acc}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'  Early stopping phase {phase_num} at epoch {epoch} '
                      f'(no val_f1 improvement for {patience} epochs)')
                break
        return epochs_no_improve, best_val_f1, best_state, best_epoch_info

    # --- Phase 1: frozen backbone, head only ---
    freeze_all_backbone(model)
    opt = build_discriminative_optimizer(model, head_lr=3e-4, backbone_lr=0)  # backbone frozen, lr irrelevant
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_per_phase)
    epochs_no_improve, best_val_f1, best_state, best_epoch_info = run_phase(
        1, opt, scheduler, epochs_no_improve, best_val_f1, best_state, best_epoch_info
    )

    # --- Phase 2: unfreeze 20% of backbone ---
    epochs_no_improve = 0
    unfreeze_fraction(model, 0.2)
    # head keeps learning fast, unfrozen backbone slice gets a real (not negligible) but smaller LR
    opt = build_discriminative_optimizer(model, head_lr=1e-4, backbone_lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_per_phase)
    epochs_no_improve, best_val_f1, best_state, best_epoch_info = run_phase(
        2, opt, scheduler, epochs_no_improve, best_val_f1, best_state, best_epoch_info
    )

    # --- Phase 3: unfreeze 1/3 of backbone ---
    epochs_no_improve = 0
    unfreeze_fraction(model, 1/3)
    opt = build_discriminative_optimizer(model, head_lr=5e-5, backbone_lr=5e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_per_phase)
    epochs_no_improve, best_val_f1, best_state, best_epoch_info = run_phase(
        3, opt, scheduler, epochs_no_improve, best_val_f1, best_state, best_epoch_info
    )

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f'/kaggle/working/{tag}.pth')
    print(f'Best checkpoint: phase {best_epoch_info["phase"]}, epoch {best_epoch_info["epoch"]}, '
          f'val_f1={best_epoch_info["val_f1"]:.4f}, val_acc={best_epoch_info["val_acc"]:.4f}')
    return model, pd.DataFrame(history), best_epoch_info


# Run it — same backbone as your best 4-channel run, for apples-to-apples comparison
model_rgb, history_rgb, best_info_rgb = run_three_phase_training_rgb(
    'resnet50v2', extra_dense=True, tag='resnet50v2_dense_RGB_ablation'
)

# Test set evaluation
criterion = nn.CrossEntropyLoss()
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate_rgb(model_rgb, test_loader_rgb, criterion)
print(f'\n[RGB-only] Test acc: {test_acc:.4f}  Test f1_macro: {test_f1:.4f}')

In [ ]:
# ============================================================
# RGB-ONLY ABLATION (3-channel, no mask channel)
# Reuses the same crops (.npy files already have RGB in ch 0:3),
# just ignores channel 4 at load time.
# ============================================================
from sklearn.metrics import f1_score, cohen_kappa_score
MEAN_3CH = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD_3CH  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def normalize_3ch(tensor):
    """tensor: (3,H,W) uint8 -> normalized float"""
    tensor = tensor.float() / 255.0
    return (tensor - MEAN_3CH) / STD_3CH


import torch.nn.functional as F



class WoundDatasetRGB(Dataset):
    """Same crops as WoundDataset4Ch, but drops the mask channel entirely."""
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        stacked = np.load(row['path'])  # (H,W,4) uint8
        label = row['grade'] - 1
        rgb = stacked[:, :, :3]  # ignore channel 4 entirely

        if self.transform:
            augmented = self.transform(image=rgb)
            rgb_t = augmented['image']  # (3,H,W) tensor from ToTensorV2
        else:
            rgb_t = torch.from_numpy(rgb.transpose(2, 0, 1))

        rgb_t = normalize_3ch(rgb_t)
        return rgb_t, label
        


# ============================================================
# 1. Updated augmentation with CoarseDropout
# ============================================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.CoarseDropout(
        max_holes=8, max_height=int(IMG_SIZE*0.1), max_width=int(IMG_SIZE*0.1),
        min_holes=2, min_height=int(IMG_SIZE*0.05), min_width=int(IMG_SIZE*0.05),
        fill_value=0, p=0.4
    ),
    ToTensorV2()
], additional_targets={'mask_ch': 'mask'})
# val_transform stays unchanged (no augmentation at eval time)


# Use the SAME manifest/splits as your current 4-channel run for a fair comparison
train_df = manifest_df[manifest_df['split']=='train']
val_df   = manifest_df[manifest_df['split']=='val']
test_df  = manifest_df[manifest_df['split']=='test']
train_ds_rgb = WoundDatasetRGB(train_df, train_transform_rgb)
val_ds_rgb   = WoundDatasetRGB(val_df, val_transform_rgb)
test_ds_rgb  = WoundDatasetRGB(test_df, val_transform_rgb)

train_loader_rgb = DataLoader(train_ds_rgb, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_rgb   = DataLoader(val_ds_rgb, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_rgb  = DataLoader(test_ds_rgb, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def freeze_all_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

# ============================================================
# 2. Smaller, more targeted unfreeze — last block only
# ============================================================
def unfreeze_last_n_params(model, n_params):
    """Unfreeze only the last n_params trainable tensors of the backbone
    (approximates 'last block only' without needing to know exact module names)."""
    params = list(model.backbone.parameters())
    for p in params[:-n_params]:
        p.requires_grad = False
    for p in params[-n_params:]:
        p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True

# Helper to inspect how many param tensors exist, so you can pick n_params sensibly.
# Run this once to calibrate:
#   model_tmp = WoundClassifier4Ch('resnet50v2', NUM_CLASSES, True)
#   print(len(list(model_tmp.backbone.parameters())))
#   for name, _ in model_tmp.backbone.named_parameters():
#       print(name)
# Then pick n_params to correspond to roughly the final stage/block.

# ============================================================
# 3. Optimizer with higher weight decay on backbone params
# ============================================================
def build_discriminative_optimizer(model, head_lr, backbone_lr,
                                    head_wd=1e-4, backbone_wd=1e-2):
    """Two param groups with independent weight decay:
    backbone gets much stronger decay to fight the overfitting we saw."""
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params = [p for p in model.head.parameters() if p.requires_grad]
    param_groups = []
    if backbone_params:
        param_groups.append({'params': backbone_params, 'lr': backbone_lr, 'weight_decay': backbone_wd})
    if head_params:
        param_groups.append({'params': head_params, 'lr': head_lr, 'weight_decay': head_wd})
    return optim.AdamW(param_groups)
# ============================================================
# 1. CORN loss function
# ============================================================
def corn_loss(logits, labels, num_classes):
    """
    CORN loss (Shi et al., 2021 - 'Conditional Ordinal Regression for Neural Networks').
    logits: (batch, num_classes-1) raw outputs
    labels: (batch,) integer grade in [0, num_classes-1]  (i.e. grade-1)
    """
    total_loss = 0.0
    batch_size = logits.size(0)
    for k in range(num_classes - 1):
        # Only samples with true label > k-1 (i.e. label >= k) are "eligible"
        # for the k-th binary task (this is what makes CORN "conditional")
        mask = (labels >= k)
        if mask.sum() == 0:
            continue
        target_k = (labels[mask] > k).float()  # is true label > k?
        logit_k = logits[mask, k]
        loss_k = F.binary_cross_entropy_with_logits(logit_k, target_k, reduction='sum')
        total_loss += loss_k
    return total_loss / batch_size


def corn_predict(logits):
    """
    Decode CORN logits into a predicted class index.
    logits: (batch, num_classes-1)
    Returns: (batch,) predicted grade index (0-based)
    """
    probs = torch.sigmoid(logits)              # P(label > k) for k=0..K-2
    preds = (probs > 0.5).sum(dim=1)            # count how many thresholds are exceeded
    return preds
BACKBONE_MAP = {
    'resnet50v2': 'resnetv2_50x1_bit.goog_in21k',

}

# ============================================================
# 2. Updated classifier head — outputs K-1 logits instead of K
# ============================================================
class WoundClassifierCORN(nn.Module):
    def __init__(self, backbone_key, num_classes=4, extra_dense=False, dense_dim=256, in_channels=3):
        super().__init__()
        backbone = timm.create_model(BACKBONE_MAP[backbone_key], pretrained=True, num_classes=0)
        if in_channels == 4:
            self.backbone = find_and_replace_first_conv(backbone)
        else:
            self.backbone = backbone
        feat_dim = self.backbone.num_features
        out_dim = num_classes - 1  # CORN: K-1 outputs
        if extra_dense:
            self.head = nn.Sequential(
                nn.Linear(feat_dim, dense_dim), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(dense_dim, out_dim)
            )
        else:
            self.head = nn.Linear(feat_dim, out_dim)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)


# ============================================================
# 3. Updated train/eval loops using CORN loss + CORN decoding
# ============================================================
def train_one_epoch_corn(model, loader, optimizer, num_classes):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, labels in loader:
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = corn_loss(logits, labels, num_classes)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds = corn_predict(logits)
        correct += (preds == labels).sum().item()
        total += x.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate_corn(model, loader, num_classes):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for x, labels in loader:
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        logits = model(x)
        loss = corn_loss(logits, labels, num_classes)
        total_loss += loss.item() * x.size(0)
        preds = corn_predict(logits)
        correct += (preds == labels).sum().item()
        total += x.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')  # ordinal-aware metric
    return total_loss/total, correct/total, f1_macro, qwk, all_preds, all_labels

# ============================================================
# 4. Simplified single-phase training (frozen backbone only,
#    per our decision to abandon phases 2/3 for now)
# ============================================================
def run_corn_training(backbone_key, extra_dense, tag, num_classes=4,
                       epochs=20, patience=7, in_channels=3):
    print(f'\n=== CORN Training {backbone_key} | extra_dense={extra_dense} | in_channels={in_channels} ===')
    model = WoundClassifierCORN(backbone_key, num_classes, extra_dense, in_channels=in_channels).to(DEVICE)

    # Frozen backbone, head-only training (phase-1-only regime, per our earlier finding)
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    history = []
    best_val_f1 = -1
    best_state = None
    best_epoch_info = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch_corn(model, train_loader_rgb, opt, num_classes)
        val_loss, val_acc, val_f1, val_qwk, _, _ = evaluate_corn(model, val_loader_rgb, num_classes)
        scheduler.step()
        history.append({
            'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc,
            'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1, 'val_qwk': val_qwk
        })
        print(f'E{epoch}: train_acc={tr_acc:.3f} val_acc={val_acc:.3f} val_f1={val_f1:.3f} val_qwk={val_qwk:.3f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch_info = {'epoch': epoch, 'val_f1': val_f1, 'val_acc': val_acc, 'val_qwk': val_qwk}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f'/kaggle/working/{tag}.pth')
    print(f'Best checkpoint: epoch {best_epoch_info["epoch"]}, val_f1={best_epoch_info["val_f1"]:.4f}, '
          f'val_acc={best_epoch_info["val_acc"]:.4f}, val_qwk={best_epoch_info["val_qwk"]:.4f}')
    return model, pd.DataFrame(history), best_epoch_info

# Run it — RGB-only, since we confirmed the mask channel doesn't help
model_corn, history_corn, best_info_corn = run_corn_training(
    'resnet50v2', extra_dense=True, tag='resnet50v2_corn', in_channels=3
)

# Test evaluation
test_loss, test_acc, test_f1, test_qwk, test_preds, test_labels = evaluate_corn(model_corn, test_loader_rgb, 4)
print(f'\n[CORN] Test acc: {test_acc:.4f}  Test f1_macro: {test_f1:.4f}  Test QWK: {test_qwk:.4f}')

# Run it — same backbone as your best 4-channel run, for apples-to-apples comparison
model_rgb, history_rgb, best_info_rgb = run_three_phase_training_rgb(
    'resnet50v2', extra_dense=True, tag='resnet50v2_dense_RGB_ablation'
)

# Test set evaluation
criterion = nn.CrossEntropyLoss()
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate_corn(model_rgb, test_loader_rgb, criterion)
print(f'\n[RGB-only] Test acc: {test_acc:.4f}  Test f1_macro: {test_f1:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd 

# Class names (change if you use different names)
class_names = ["Grade 1", "Grade 2", "Grade 3", "Grade 4"]

# Compute confusion matrix
cm = confusion_matrix(test_labels, test_preds)

# Plot
plt.figure(figsize=(7,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Confusion Matrix - RGB-only Ablation", fontsize=14)
plt.tight_layout()
plt.show()

# Print classification report
print(classification_report(
    test_labels,
    test_preds,
    target_names=class_names,
    digits=4
))

In [ ]:
def visualize_full_pipeline(img_path, model):
    # -------------------------
    # Original image
    # -------------------------
    orig = cv2.imread(img_path)
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)

    # -------------------------
    # Segmentation + crop
    # -------------------------
    crop_img, crop_mask, fell_back = predict_mask_and_crop(img_path)

    crop_img_r = cv2.resize(crop_img, (IMG_SIZE, IMG_SIZE))

    # -------------------------
    # RGB ONLY (3 channels)
    # -------------------------
    tensor = torch.from_numpy(crop_img_r.transpose(2, 0, 1))
    tensor = normalize_3ch(tensor).unsqueeze(0).to(DEVICE)

    # -------------------------
    # Prediction
    # -------------------------
    model.eval()

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_class = np.argmax(probs)
    pred_grade = pred_class + 1
    confidence = probs[pred_class]

    # -------------------------
    # Visualization
    # -------------------------
    fig, axes = plt.subplots(1, 4, figsize=(20,5))

    axes[0].imshow(orig)
    axes[0].set_title(f"1. Original\n{orig.shape[1]}×{orig.shape[0]}")
    axes[0].axis("off")

    axes[1].imshow(crop_mask, cmap="gray")
    axes[1].set_title(f"2. Segmentation Mask\nfallback={fell_back}")
    axes[1].axis("off")

    axes[2].imshow(crop_img)
    axes[2].set_title("3. Cropped wound")
    axes[2].axis("off")

    color = "green" if confidence > 0.5 else "orange"

    axes[3].imshow(crop_img_r)
    axes[3].set_title(
        f"4. Prediction\nGrade {pred_grade}\n{confidence:.1%}",
        color=color
    )
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"File: {os.path.basename(img_path)}")
    print(f"Fallback used: {fell_back}")

    for i, p in enumerate(probs):
        print(f"Grade {i+1}: {p:.2%}")

    print(f"\nFinal prediction: Grade {pred_grade} ({confidence:.2%})")

    return pred_grade, confidence, fell_back

In [ ]:
best_model_path = '/kaggle/working/resnet50v2_dense_RGB_ablation.pth'
backbone_key = 'resnet50v2'
extra_dense = True

best_model = WoundClassifierRGB(backbone_key, NUM_CLASSES, extra_dense).to(DEVICE)
state_dict = torch.load(best_model_path, map_location=DEVICE) 
best_model.load_state_dict(state_dict)
best_model.eval()

print(type(best_model))  # should now print <class '__main__.WoundClassifier4Ch'>

In [ ]:

def run_random_batch_check(n_samples=4, split='test'):
    all_files = []
    for grade, dir_path in GRADE_DIRS[split].items():
        files = glob.glob(f'{dir_path}/*.jpg') + glob.glob(f'{dir_path}/*.png')
        all_files.extend([(fp, grade) for fp in files])

    samples = random.sample(all_files, min(n_samples, len(all_files)))

    for fp, true_grade in samples:
        print(f'\n{"="*70}\nTrue grade (from folder): Grade {true_grade}\n{"="*70}')
        pred_grade, confidence, fell_back = visualize_full_pipeline(fp, best_model)
        match = 'CORRECT' if pred_grade == true_grade else 'MISMATCH'
        print(f'>>> {match} — true=Grade {true_grade}, predicted=Grade {pred_grade}\n')

run_random_batch_check(n_samples=4, split='test')